# ArchiGAN-SL — Génération de plans 2D par cGAN piloté par prompt, entraîné en Split Learning

**Auteur :** Kenny Tshibangu Ntumba — Université Nouveaux Horizons, Lubumbashi
**Version :** v2 (2026) — fait suite au prototype v1 du mémoire de Master (2025)

Pipeline d'inférence :

```
prompt texte → parser → contraintes (−1 = libre) → condition [valeurs | masque]
            → cGAN G(z, c) → programme = 9 comptes de pièces + 6 variables de graphe
            → SNAP (contraintes explicites forcées)
            → graphe de bulles → placement → portes en arbre + entrée → rendu architectural
```

Pipeline d'entraînement collaboratif (Split Learning en U, simulé dans un seul processus) :

```
 CLIENT k (agence, données privées)              SERVEUR (partagé)
 ┌──────────────────────────────┐   e = E_k(c)   ┌────────────────────────┐
 │ conditions c, vecteurs réels v│ ─────────────▶ │ G_s(z, e) → v̂          │
 │ E_k : encodeur de condition   │ ◀───────────── │                        │
 │ B_k : bas du discriminateur   │   s = B_k(v,c) │ T : haut du discrim.   │
 │ perte de contraintes (locale) │ ─────────────▶ │ perte adversariale     │
 └──────────────────────────────┘  ◀── gradients └────────────────────────┘
```
Ni `v` ni `c` ne quittent le client : seuls transitent les *smashed data* (`e`, `s`), les vecteurs générés `v̂` et les gradients.

### Différences avec la v1 (mémoire 2025)
| | v1 (mémoire) | v2 (ce notebook) |
|---|---|---|
| Sortie de G | vecteur de pièces → placement → image 512 px jugée par un D convolutif | programme de 15 entiers (pièces + **graphe de bulles**) jugé directement par D (MLP) |
| Conditionnement | aucun (prompt traité hors du GAN) | cGAN : condition = [valeurs normalisées, masque] |
| Pertes | adversariale + adjacence + SSIM (non différentiables à travers `place_rooms`) | adversariale + λ · violation des contraintes (différentiable) |
| Données | images de plans réels | 20 000 programmes (pièces + topologie) validés par le placement |
| Prompt | expressions régulières sur les pièces | pièces, typologies, négations, surface, **structure** (« suite parentale », « cuisine fermée », « WC séparés »…) |
| Split Learning | conceptuel | implémenté (U-shaped, K clients, comptage des octets échangés) |
| Placement | glouton, 1 essai, une porte sur chaque contact, pas d'entrée | **v3** : graphe de bulles (séjour → couloir → chambres → salle d'eau…), position la plus compacte, portes en arbre (une porte par pièce privée, règles interdites), porte d'entrée, tailles adaptées à la surface du prompt |
| Évaluation | pertes, SSIM, connectivité | respect du prompt, plausibilité, diversité, portes légales, temps, baselines, ablations |

### Organisation
| Cellule | Contenu |
|---|---|
| 0 | Configuration (Colab / local, graines, hyperparamètres) |
| 1–3 | Fondations : vocabulaire, placement v2, rendu, compaction |
| 3b | Placement v3 (graphe de bulles, portes en arbre, entrée, surface) |
| 4 | Parser de prompt (pièces, typologies, négations, surface / dimensions) |
| 5 | Métriques de qualité des plans |
| 6 | Briques du cGAN (dataset conditionnel, G, D, pertes, entraînement, génération) |
| 7 | Dataset de programmes + découpage apprentissage / test |
| 8 | Entraînement centralisé (référence) |
| 9 | Le prompter `plan()` + figure d'exemples |
| 10 | Évaluation quantitative (méthode proposée vs baselines) |
| 11 | Ablation (λ, snap) |
| 12 | Split Learning : implémentation |
| 13 | Split Learning : expériences (SplitFed, SL, local seul, centralisé) |
| 14 | Fuite d'information des *smashed data* (attaque d'inversion) |
| 15 | Export des résultats (JSON, figures, macros LaTeX pour l'article) |

> Mettre `QUICK = True` dans la cellule 0 pour un test de bout en bout en quelques minutes.

## Cellule 0 — Configuration

In [ ]:
# =====================================================================
# CELLULE 0 - CONFIGURATION
# =====================================================================
# Tous les reglages sont ici. Rien d'autre a modifier pour relancer.
#   QUICK = True  -> test de bout en bout (petit dataset, peu d'epoques)
#   RUN_*         -> active / desactive les experiences longues
# =====================================================================

import os, sys, json, time, random
import numpy as np
import torch

QUICK = os.environ.get("ARCHIGAN_QUICK", "0") == "1"   # ou forcer True/False

RUN_ABLATION = True    # cellule 11 (~15 min GPU)
RUN_SPLIT    = True    # cellules 12-14 (~30 min GPU)
RUN_NOISE    = True    # SplitFed avec bruit sur les smashed data (defense)

SEED = 42

CFG = {
    "n_vectors":      1500 if QUICK else 20000,
    "test_ratio":     0.10,
    "batch_size":     128,
    "latent_dim":     32,
    "lr":             2e-4,
    "lambda":         15.0,
    "epochs":         4 if QUICK else 200,
    "abl_epochs":     2 if QUICK else 60,
    "abl_lambdas":    [0.0, 1.0, 5.0, 15.0, 50.0],
    "sl_clients":     3,
    "sl_epochs":      3 if QUICK else 100,
    "sl_noise_std":   0.5,
    "eval_variants":  4 if QUICK else 50,
    "eval_test_conds": 300 if QUICK else 2000,
    "grid_size":      60,
    "layout_tries":   5,      # essais de placement (best-of-k), 1 = v1
    "placement_fallback": True,   # repli si parents legaux satures, False = v1
    "placement":      "v3",   # "v3" graphe de bulles + portes en arbre, "v2" glouton
}

# --- Environnement : Colab (Drive) ou local --------------------------
try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/ArchiGAN/cgan/"
else:
    BASE_DIR = os.path.abspath("./archigan_runs/")

if QUICK:
    BASE_DIR = os.path.join(BASE_DIR, "quick")
MODEL_DIR  = os.path.join(BASE_DIR, "checkpoints")
EXPORT_DIR = os.path.join(BASE_DIR, "export")
for d in (BASE_DIR, MODEL_DIR, EXPORT_DIR):
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(s=SEED):
    """Reproductibilite : random (placement), numpy, torch."""
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


set_seed()
RESULTS = {"config": CFG, "quick": QUICK, "device": str(device),
           "torch": torch.__version__,
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}

print(f"Colab: {IN_COLAB} | device: {device} | QUICK: {QUICK}")
print(f"Sorties: {BASE_DIR}")

## Cellule 1 — Fondations : vocabulaire, placement glouton, conformité d'adjacence

In [ ]:
# =====================================================================
# CELLULE 1 - FONDATIONS DU PROJET (dicts + fonctions procedurales)
# =====================================================================
# But : poser TOUS les elements non-ML dont le GAN aura besoin :
#   - Le vocabulaire des pieces (room_ids, room_sizes, room_adjacencies)
#   - Le pipeline procedural de placement (place_rooms, ...)
#   - L'evaluation de la qualite (evaluate_adjacency_compliance)
#
# Cette cellule ne contient PAS de reseau de neurones - c'est le socle
# metier. Les cellules suivantes s'appuient dessus.
# =====================================================================

import os
import random
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Vocabulaire des pieces ---------------------------------------
# Meme mapping que le prototype v1 du memoire (compatibilite).
room_ids = {
    "vide": 0, "salon": 1, "chambre": 2, "douche": 3, "toilettes": 4,
    "couloir": 5, "bureau": 6, "cuisine": 8, "porte": 9,
    "exterieur": 11, "balcon": 12, "dressing": 13
}
id_to_room = {v: k for k, v in room_ids.items()}

# Tailles (largeur, hauteur) en cellules de grille - pas en pixels
room_sizes = {
    0: (8, 8),      # vide
    1: (14, 16),    # salon (grand)
    2: (10, 12),    # chambre
    3: (6, 8),      # douche
    4: (4, 6),      # toilettes
    5: (3, 20),     # couloir (long et etroit)
    6: (8, 10),     # bureau
    8: (10, 10),    # cuisine
    9: (1, 1),      # porte
    11: (1, 1),     # exterieur
    12: (6, 10),    # balcon
    13: (4, 6)      # dressing
}

# Regles d'adjacence : quelles pieces peuvent/doivent etre voisines
# Chaque tuple = (nom_voisin, probabilite qu'on tente l'adjacence)
room_adjacencies = {
    # v3 : dans les petits logements (studio, T2) le sejour sert de
    # degagement -> chambre, douche, WC et bureau peuvent s'y ouvrir.
    # (WC sur le sejour reste INTERDIT des qu'un couloir existe, cf.
    #  is_forbidden, cellule 3b)
    "salon":     [("cuisine", 0.7), ("couloir", 0.9), ("balcon", 0.7), ("exterieur", 0.8),
                  ("chambre", 0.3), ("douche", 0.2), ("bureau", 0.4), ("toilettes", 0.2)],
    "couloir":   [("salon", 0.9), ("chambre", 0.95), ("cuisine", 0.7),
                  ("douche", 0.8), ("bureau", 0.6), ("toilettes", 0.5), ("dressing", 0.4)],
    "chambre":   [("couloir", 0.9), ("douche", 0.7), ("dressing", 0.6), ("balcon", 0.3)],
    "cuisine":   [("salon", 0.7), ("couloir", 0.7), ("balcon", 0.2)],
    "douche":    [("chambre", 0.7), ("couloir", 0.8)],
    "toilettes": [("couloir", 0.9), ("douche", 0.4)],
    "bureau":    [("couloir", 0.8), ("salon", 0.4)],
    "dressing":  [("chambre", 0.8), ("couloir", 0.5)],
    "balcon":    [("salon", 0.7), ("chambre", 0.4), ("cuisine", 0.2)],
    "exterieur": [("salon", 0.9)]
}


# v2 : repli sur les autres pieces quand les parents legaux sont satures
# (False = comportement exact du prototype v1)
PLACEMENT_FALLBACK = CFG.get("placement_fallback", True) if "CFG" in globals() else True


# --- 2. Utilitaires de placement -------------------------------------

def get_adjacent_rooms(room_name):
    """Retourne la liste des pieces adjacentes possibles pour room_name,
    en filtrant selon les probabilites (donc resultat stochastique)."""
    possible = room_adjacencies.get(room_name, [])
    return [r[0] for r in possible if random.random() < r[1]]


def is_valid_placement(grid, x, y, w, h, ignore_ids=[]):
    """Verifie qu'un rectangle (x, y, w, h) peut etre pose sur la grille
    sans depasser les bords ni empieter sur une piece existante."""
    h_grid, w_grid = grid.shape
    if x < 0 or y < 0 or x + w > w_grid or y + h > h_grid:
        return False
    for r in range(y, y + h):
        for c in range(x, x + w):
            if grid[r, c] != room_ids["vide"] and grid[r, c] not in ignore_ids:
                return False
    return True


def place_room_and_door(grid, new_room_pos, parent_room_pos):
    """Essaye de percer une porte entre deux pieces adjacentes.
    Retourne (succes, dict_info_porte)."""
    nx, ny, nw, nh = new_room_pos
    px, py, pw, ph = parent_room_pos

    if nx == px + pw:                    # new a droite de parent
        overlap_start, overlap_end = max(ny, py), min(ny + nh, py + ph)
        orientation, side = 'vertical', 'right'
        fixed_new, fixed_parent = nx, px + pw - 1
    elif px == nx + nw:                  # new a gauche de parent
        overlap_start, overlap_end = max(ny, py), min(ny + nh, py + ph)
        orientation, side = 'vertical', 'left'
        fixed_new, fixed_parent = nx + nw - 1, px
    elif ny == py + ph:                  # new en dessous de parent
        overlap_start, overlap_end = max(nx, px), min(nx + nw, px + pw)
        orientation, side = 'horizontal', 'bottom'
        fixed_new, fixed_parent = ny, py + ph - 1
    elif py == ny + nh:                  # new au dessus de parent
        overlap_start, overlap_end = max(nx, px), min(nx + nw, px + pw)
        orientation, side = 'horizontal', 'top'
        fixed_new, fixed_parent = ny + nh - 1, py
    else:
        return False, None

    door_size = 2
    if overlap_start >= overlap_end - door_size + 1:
        return False, None

    door_pos = random.randint(overlap_start, overlap_end - door_size)

    # On marque les pixels de porte dans la grille pour que le rendu
    # puisse creer l'ouverture visuelle plus tard
    if orientation == 'vertical':
        for r in range(door_pos, door_pos + door_size):
            if 0 <= r < grid.shape[0]:
                if 0 <= fixed_new < grid.shape[1]:
                    grid[r, fixed_new] = room_ids["porte"]
                if 0 <= fixed_parent < grid.shape[1]:
                    grid[r, fixed_parent] = room_ids["porte"]
        door_info = {'pos': (fixed_new, door_pos), 'size': door_size,
                     'orientation': orientation, 'side': side}
    else:  # horizontal
        for c in range(door_pos, door_pos + door_size):
            if 0 <= c < grid.shape[1]:
                if 0 <= fixed_new < grid.shape[0]:
                    grid[fixed_new, c] = room_ids["porte"]
                if 0 <= fixed_parent < grid.shape[0]:
                    grid[fixed_parent, c] = room_ids["porte"]
        door_info = {'pos': (door_pos, fixed_new), 'size': door_size,
                     'orientation': orientation, 'side': side}

    return True, door_info


def place_rooms(grid, pieces_to_place):
    """Place les pieces sur la grille en tenant compte des adjacences.
    C'est l'algorithme glouton central du projet.

    Args:
      grid : np.array 2D initialisee a zero
      pieces_to_place : dict {name: count}, ex {'salon': 1, 'chambre': 2, ...}

    Returns:
      grid modifiee, liste des pieces placees, liste des portes creees.
    """
    placed_rooms, placed_doors = [], []
    h, w = grid.shape

    # On construit la liste ordonnee des pieces a placer (par priorite)
    rooms_list = []
    for name, count in pieces_to_place.items():
        if name in ['vide', 'porte', 'exterieur']:
            continue
        rid = room_ids[name]
        priority = 0 if name == 'salon' else (1 if name == 'couloir'
                    else (2 if name in ['cuisine', 'chambre'] else 3))
        w_room, h_room = room_sizes[rid]
        orientations = [(w_room, h_room)]
        if name == 'couloir' and w_room != h_room:
            orientations.append((h_room, w_room))
        for _ in range(count):
            rooms_list.append({'id': rid, 'name': name,
                               'priority': priority,
                               'orientations': orientations})
    rooms_list.sort(key=lambda x: x['priority'])

    # Placement iteratif
    for room_to_place in rooms_list:
        room_id = room_to_place['id']
        room_name = room_to_place['name']
        placed = False

        # 1ere piece : au centre de la grille
        if not placed_rooms:
            for room_w, room_h in room_to_place['orientations']:
                cx, cy = (w - room_w) // 2, (h - room_h) // 2
                if is_valid_placement(grid, cx, cy, room_w, room_h):
                    grid[cy:cy+room_h, cx:cx+room_w] = room_id
                    placed_rooms.append({
                        'id': room_id, 'pos': (cx, cy, room_w, room_h),
                        'name': room_name, 'unique_id': f"{room_name}_0"
                    })
                    placed = True
                    break
            continue

        # Autres pieces : on cherche un parent adjacent legal
        possible_parents = [p for p in placed_rooms
                            if room_name in get_adjacent_rooms(p['name'])
                            or p['name'] in get_adjacent_rooms(room_name)]
        if not possible_parents:
            possible_parents = placed_rooms.copy()
        random.shuffle(possible_parents)
        # Repli (v2) : si tous les parents "legaux" sont satures, on tente
        # les autres pieces plutot que d'abandonner la piece.
        if PLACEMENT_FALLBACK:
            others = [p for p in placed_rooms if p not in possible_parents]
            random.shuffle(others)
            possible_parents = possible_parents + others

        for parent in possible_parents:
            px, py, pw, ph = parent['pos']
            sides = ['top', 'right', 'bottom', 'left']
            random.shuffle(sides)
            for side in sides:
                for room_w, room_h in room_to_place['orientations']:
                    nx, ny = -1, -1
                    min_overlap = 2
                    if side == 'top':
                        ny = py - room_h
                        s, e = (max(0, px - room_w + min_overlap),
                                min(w - room_w, px + pw - min_overlap))
                    elif side == 'right':
                        nx = px + pw
                        s, e = (max(0, py - room_h + min_overlap),
                                min(h - room_h, py + ph - min_overlap))
                    elif side == 'bottom':
                        ny = py + ph
                        s, e = (max(0, px - room_w + min_overlap),
                                min(w - room_w, px + pw - min_overlap))
                    elif side == 'left':
                        nx = px - room_w
                        s, e = (max(0, py - room_h + min_overlap),
                                min(h - room_h, py + ph - min_overlap))
                    if s > e:
                        continue
                    for _ in range(10):
                        cur_nx = random.randint(s, e) if side in ['top', 'bottom'] else nx
                        cur_ny = random.randint(s, e) if side in ['left', 'right'] else ny
                        cur_nx = max(0, min(w - room_w, cur_nx))
                        cur_ny = max(0, min(h - room_h, cur_ny))
                        if is_valid_placement(grid, cur_nx, cur_ny, room_w, room_h):
                            grid[cur_ny:cur_ny+room_h,
                                 cur_nx:cur_nx+room_w] = room_id
                            count_same = len([p for p in placed_rooms
                                              if p['name'] == room_name])
                            info = {
                                'id': room_id,
                                'pos': (cur_nx, cur_ny, room_w, room_h),
                                'name': room_name,
                                'unique_id': f"{room_name}_{count_same}"
                            }
                            placed_rooms.append(info)
                            ok, door = place_room_and_door(grid, info['pos'],
                                                            parent['pos'])
                            if ok:
                                door['connected_rooms'] = (parent['unique_id'],
                                                            info['unique_id'])
                                placed_doors.append(door)
                            placed = True
                            break
                    if placed: break
                if placed: break
            if placed: break

    # Marquer la bordure exterieure de la grille
    for y in range(h):
        for x in range(w):
            if x == 0 or x == w-1 or y == 0 or y == h-1:
                if grid[y, x] == room_ids["vide"]:
                    grid[y, x] = room_ids["exterieur"]

    return grid, placed_rooms, placed_doors


# --- 3. Evaluation de la qualite d'un plan ---------------------------

def evaluate_adjacency_compliance(placed_rooms, placed_doors):
    """Retourne un float entre 0 et 1 : proportion de pieces qui ont
    au moins une adjacence LEGALE (via une porte) selon room_adjacencies.
    """
    if not placed_rooms:
        return 0.0
    # Graphe des connexions via les portes
    graph = {r['unique_id']: [] for r in placed_rooms}
    for d in placed_doors:
        if 'connected_rooms' in d:
            r1, r2 = d['connected_rooms']
            if r1 in graph and r2 in graph:
                graph[r1].append(r2)
                graph[r2].append(r1)
    # Comptage des pieces conformes
    compliant = 0
    for room in placed_rooms:
        name = room['name']
        if name not in room_adjacencies:
            compliant += 1
            continue
        neighbor_names = []
        for cid in graph.get(room['unique_id'], []):
            cr = next((r for r in placed_rooms if r['unique_id'] == cid), None)
            if cr:
                neighbor_names.append(cr['name'])
        allowed = [x[0] for x in room_adjacencies[name]]
        if any(a in allowed for a in neighbor_names):
            compliant += 1
    return compliant / len(placed_rooms)


# --- 4. Test rapide pour verifier que tout fonctionne ----------------
print("Test rapide de la cellule 1...")
random.seed(42)
np.random.seed(42)
test_pieces = {"salon": 1, "cuisine": 1, "chambre": 2, "couloir": 1,
                "douche": 1, "toilettes": 1}
grid_test = np.zeros((60, 60), dtype=int)
grid_test, placed, doors = place_rooms(grid_test, test_pieces)
score = evaluate_adjacency_compliance(placed, doors)
print(f"  Pieces placees : {len(placed)}/{sum(test_pieces.values())}")
print(f"  Portes creees  : {len(doors)}")
print(f"  Score adjacence: {score:.2f}")
print("\n[Cellule 1 OK] Fondations pretes.")

## Cellule 2 — Rendu architectural noir & blanc

In [ ]:
# =====================================================================
# CELLULE 2 - RENDU ARCHITECTURAL NOIR & BLANC
# =====================================================================
"""
archi_render.py  --  Rendu minimaliste noir & blanc d'un plan 2D.

Noir et blanc. Murs pleins, portes a convention architecturale (vantail 90deg
+ arc de debattement), nom de piece seul. Echelle : 1 cellule = 30 cm.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

CELL_CM = 30
INK   = "#111111"   # noir principal
PAPER = "#FFFFFF"
FILL  = "#FFFFFF"   # interieur des pieces (blanc)

def render_architectural_plan(grid, placed_rooms, placed_doors,
                              room_ids_map, id_to_room_map,
                              title="Plan", save_path=None, show=True,
                              ax=None, title_size=13):
    """Dessine le plan. Si `ax` est fourni, dessine dans cet axe (figure
    composite) sans sauvegarder ni afficher."""
    H, W = grid.shape

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(11, 11), facecolor=PAPER)
    else:
        fig = ax.figure
    ax.set_facecolor(PAPER)

    # Cadrage serre sur la zone batie
    ys, xs = np.where(grid != room_ids_map["exterieur"])
    if len(xs):
        m = 2
        x0, x1 = xs.min() - m, xs.max() + 1 + m
        y0, y1 = ys.min() - m, ys.max() + 1 + m
        span = max(x1 - x0, y1 - y0)
        # Centrer dans un carre pour ne pas deformer
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
        x0, x1 = cx - span / 2, cx + span / 2
        y0, y1 = cy - span / 2, cy + span / 2
    else:
        x0, x1, y0, y1 = 0, W, 0, H
    ax.set_xlim(x0, x1); ax.set_ylim(y1, y0)
    ax.set_aspect('equal'); ax.axis('off')

    # 1) Interieur blanc des pieces (masque le fond)
    for r in placed_rooms:
        x, y, w, h = r['pos']
        ax.add_patch(Rectangle((x, y), w, h, facecolor=FILL,
                     edgecolor='none', zorder=1))

    # 2) Cloisons : contour de chaque piece (trait moyen)
    for r in placed_rooms:
        x, y, w, h = r['pos']
        ax.add_patch(Rectangle((x, y), w, h, facecolor='none',
                     edgecolor=INK, linewidth=1.6, zorder=3,
                     joinstyle='miter'))

    # 3) Murs porteurs : bords donnant sur l'exterieur (trait epais)
    def ext(rr, cc):
        for a in rr:
            for b in cc:
                if 0 <= a < H and 0 <= b < W:
                    if grid[a, b] == room_ids_map["exterieur"]:
                        return True
                else:
                    return True
        return False

    for r in placed_rooms:
        x, y, w, h = r['pos']
        if ext([y-1], range(x, x+w)):
            ax.plot([x, x+w], [y, y], color=INK, lw=5, zorder=4,
                    solid_capstyle='projecting')
        if ext([y+h], range(x, x+w)):
            ax.plot([x, x+w], [y+h, y+h], color=INK, lw=5, zorder=4,
                    solid_capstyle='projecting')
        if ext(range(y, y+h), [x-1]):
            ax.plot([x, x], [y, y+h], color=INK, lw=5, zorder=4,
                    solid_capstyle='projecting')
        if ext(range(y, y+h), [x+w]):
            ax.plot([x+w, x+w], [y, y+h], color=INK, lw=5, zorder=4,
                    solid_capstyle='projecting')

    # 4) Portes : convention archi = ouverture nette + vantail a 90deg + arc
    import math
    def door(hinge, along, openv, size):
        """hinge: point du gond. along: vecteur unitaire le long du mur
        (direction ouverture du vantail au repos = ferme). openv: vecteur
        unitaire perpendiculaire, cote ou la porte s'ouvre."""
        ax_, ay = along
        ox, oy = openv
        # Vantail dessine OUVERT a 90deg -> il pointe selon openv, longueur=size
        leaf_end = (hinge[0] + ox * size, hinge[1] + oy * size)
        ax.plot([hinge[0], leaf_end[0]], [hinge[1], leaf_end[1]],
                color=INK, lw=1.8, zorder=7)
        # Arc de debattement : du vantail ouvert (openv) vers le mur (along)
        pts = []
        for k in range(25):
            t = (k / 24) * (math.pi / 2)
            # interpolation angulaire de openv (t=0) vers along (t=pi/2)
            vx = math.cos(t) * ox + math.sin(t) * ax_
            vy = math.cos(t) * oy + math.sin(t) * ay
            pts.append((hinge[0] + vx * size, hinge[1] + vy * size))
        ax.plot([p[0] for p in pts], [p[1] for p in pts],
                color=INK, lw=0.7, zorder=7)

    # Helper : une cellule appartient-elle a une piece (pas vide/exterieur) ?
    def is_room_cell(cc, rr):
        if 0 <= rr < H and 0 <= cc < W:
            v = grid[rr, cc]
            return v != room_ids_map["exterieur"] and v != room_ids_map["vide"]
        return False

    for d in placed_doors:
        gc, gr = d['pos']; ds = d.get('size', 2)

        if d['orientation'] == 'vertical':
            # Mur vertical. Le 'fixed' de la porte = x du bord partage.
            wx = gc
            # Trou dans le mur
            ax.plot([wx, wx], [gr, gr+ds], color=PAPER, lw=8, zorder=6,
                    solid_capstyle='butt')
            # Y a-t-il une piece a gauche (wx-1) et/ou a droite (wx) du mur ?
            mid = gr + ds // 2
            room_left = is_room_cell(wx - 1, mid)
            room_right = is_room_cell(wx, mid)
            # Le vantail s'ouvre vers une piece. On prefere la pièce de
            # gauche par convention ; sinon droite.
            if room_left:
                openv = (-1, 0)
            elif room_right:
                openv = (1, 0)
            else:
                openv = (-1, 0)
            # Gond : on le met en haut ou en bas selon la place verticale.
            # On regarde si au-dessus (gr-1) ou en-dessous (gr+ds) c'est libre
            # dans la piece d'ouverture, pour eviter que l'arc sorte.
            open_col = wx - 1 if openv[0] < 0 else wx
            up_ok = is_room_cell(open_col, gr - 1)
            down_ok = is_room_cell(open_col, gr + ds)
            if down_ok:
                hinge = (wx, gr + ds); along = (0, -1)
            else:
                hinge = (wx, gr); along = (0, 1)
            door(hinge, along, openv, ds)

        else:
            wy = gr
            ax.plot([gc, gc+ds], [wy, wy], color=PAPER, lw=8, zorder=6,
                    solid_capstyle='butt')
            mid = gc + ds // 2
            room_up = is_room_cell(mid, wy - 1)
            room_down = is_room_cell(mid, wy)
            if room_down:
                openv = (0, 1)
            elif room_up:
                openv = (0, -1)
            else:
                openv = (0, 1)
            open_row = wy - 1 if openv[1] < 0 else wy
            left_ok = is_room_cell(gc - 1, open_row)
            right_ok = is_room_cell(gc + ds, open_row)
            if right_ok:
                hinge = (gc, wy); along = (1, 0)
            else:
                hinge = (gc + ds, wy); along = (-1, 0)
            door(hinge, along, openv, ds)

    # 4b) Fenetres (murs exterieurs) et portes-fenetres sur balcon :
    #     tableau dans l'epaisseur du mur, vitrage au milieu ; chassis haut
    #     (salle de bain, WC) en tirets
    names = {r['unique_id']: r['name'] for r in placed_rooms}
    wins = list(globals()['windows_for'](placed_rooms, placed_doors)) if 'windows_for' in globals() else []
    wins += [dict(d, high=False) for d in placed_doors
             if any(names.get(u) == 'balcon' for u in d['connected_rooms'])]
    for wd in wins:
        a, b = wd['pos']; s_ = wd['size']; t = 0.3
        ls = '--' if wd.get('high') else '-'
        if wd['orientation'] == 'vertical':
            ax.add_patch(Rectangle((a - t, b), 2 * t, s_, facecolor=FILL, edgecolor=INK,
                                   lw=0.7, ls=ls, zorder=6.5))
            ax.plot([a, a], [b, b + s_], color=INK, lw=0.7, ls=ls, zorder=6.6)
        else:
            ax.add_patch(Rectangle((a, b - t), s_, 2 * t, facecolor=FILL, edgecolor=INK,
                                   lw=0.7, ls=ls, zorder=6.5))
            ax.plot([a, a + s_], [b, b], color=INK, lw=0.7, ls=ls, zorder=6.6)

    # 5) Nom de piece seul
    # (texte vertical si la piece est trop etroite pour le nom)
    for r in placed_rooms:
        x, y, w, h = r['pos']
        name = r['name'].upper()
        vertical = w < 0.9 * len(name) and h > w
        size = 8.5 if max(w, h) >= 0.9 * len(name) else 6.5
        ax.text(x + w/2, y + h/2, name, ha='center', va='center',
                fontsize=size, color=INK, zorder=10, family='monospace',
                rotation=90 if vertical else 0)

    # Titre discret
    n_ch = sum(1 for r in placed_rooms if r['name'] == 'chambre')
    typo = f"T{n_ch+1}" if n_ch > 0 else "STUDIO"
    ax.set_title(f"{title.upper()}   —   {typo}",
                 fontsize=title_size, color=INK, family='monospace', pad=18)

    if not own_fig:
        return fig
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=PAPER)
        if isinstance(save_path, str):
            print(f"Plan sauve : {save_path}")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


print("[Cellule 2 OK] Rendu pret.")

## Cellule 3 — Placement compact v2 (compaction, recentrage, portes, connexité)

In [ ]:
# =====================================================================
# CELLULE 3 - PLACEMENT COMPACT (compaction + recentrage + portes)
# =====================================================================
"""
place_rooms_v2.py -- Ameliorations de placement : compaction + recentrage.

Ces fonctions s'appliquent APRES place_rooms() existant. Elles prennent
la liste placed_rooms et la retravaillent, puis on reconstruit la grille
et les portes proprement.

Principe :
  1. compact_rooms  : pousse iterativement chaque piece vers le centre de
     masse pour coller les trous (facon "gravite douce").
  2. recenter_rooms : translate tout l'ensemble pour centrer la bounding box
     dans la grille.
  3. rebuild_grid_and_doors : redessine la grille + recalcule les portes
     entre pieces devenues adjacentes.
"""

import numpy as np
import random


def rects_overlap(a, b):
    """a, b = (x, y, w, h). True si les deux rectangles se chevauchent."""
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    return not (ax + aw <= bx or bx + bw <= ax or
                ay + ah <= by or by + bh <= ay)


def can_place_at(room, new_pos, others, grid_w, grid_h):
    """Verifie qu'une piece a new_pos ne sort pas de la grille et ne
    chevauche aucune autre piece."""
    nx, ny, w, h = new_pos
    if nx < 0 or ny < 0 or nx + w > grid_w or ny + h > grid_h:
        return False
    for o in others:
        if rects_overlap(new_pos, o['pos']):
            return False
    return True


def compact_rooms(placed_rooms, grid_w, grid_h, iterations=200):
    """Pousse chaque piece d'un pas vers le centre de masse global,
    tant que ca ne cree pas de collision. Colle les trous."""
    rooms = [dict(r) for r in placed_rooms]  # copie

    for _ in range(iterations):
        # Centre de masse de l'ensemble
        cx = np.mean([r['pos'][0] + r['pos'][2] / 2 for r in rooms])
        cy = np.mean([r['pos'][1] + r['pos'][3] / 2 for r in rooms])

        moved = False
        # On traite les pieces de la plus eloignee du centre a la plus proche
        order = sorted(range(len(rooms)), key=lambda i: -(
            (rooms[i]['pos'][0] + rooms[i]['pos'][2]/2 - cx) ** 2 +
            (rooms[i]['pos'][1] + rooms[i]['pos'][3]/2 - cy) ** 2))

        for i in order:
            r = rooms[i]
            x, y, w, h = r['pos']
            rc_x, rc_y = x + w / 2, y + h / 2
            # Direction vers le centre (pas de 1 cellule sur l'axe dominant)
            dx = np.sign(cx - rc_x)
            dy = np.sign(cy - rc_y)
            others = [rooms[j] for j in range(len(rooms)) if j != i]

            # Essayer de bouger sur x puis sur y (mouvement Manhattan)
            for (mx, my) in [(dx, 0), (0, dy)]:
                if mx == 0 and my == 0:
                    continue
                new_pos = (int(x + mx), int(y + my), w, h)
                if can_place_at(r, new_pos, others, grid_w, grid_h):
                    r['pos'] = new_pos
                    x, y = new_pos[0], new_pos[1]
                    moved = True

        if not moved:
            break  # equilibre atteint

    return rooms


def recenter_rooms(placed_rooms, grid_w, grid_h):
    """Translate l'ensemble pour centrer la bounding box dans la grille."""
    xs0 = min(r['pos'][0] for r in placed_rooms)
    ys0 = min(r['pos'][1] for r in placed_rooms)
    xs1 = max(r['pos'][0] + r['pos'][2] for r in placed_rooms)
    ys1 = max(r['pos'][1] + r['pos'][3] for r in placed_rooms)

    bbox_w = xs1 - xs0
    bbox_h = ys1 - ys0
    off_x = (grid_w - bbox_w) // 2 - xs0
    off_y = (grid_h - bbox_h) // 2 - ys0

    out = []
    for r in placed_rooms:
        x, y, w, h = r['pos']
        nr = dict(r)
        nr['pos'] = (x + off_x, y + off_y, w, h)
        out.append(nr)
    return out


def rebuild_grid_and_doors(placed_rooms, grid_w, grid_h,
                            room_ids_map, place_door_fn, adjacencies):
    """Reconstruit la grille depuis les positions finales et recalcule
    les portes. Garantit que CHAQUE piece a au moins une porte et que
    l'ensemble est connexe (toute piece atteignable depuis le salon)."""
    grid = np.zeros((grid_h, grid_w), dtype=int)
    for r in placed_rooms:
        x, y, w, h = r['pos']
        grid[y:y+h, x:x+w] = r['id']

    # --- 1. Recenser TOUTES les adjacences reelles (partage de bord >=2) ---
    # Pour chaque paire adjacente on garde : orientation, side, segment,
    # taille de recouvrement, et si l'adjacence est "legale".
    adj_pairs = []  # (i, j, info)
    for i, a in enumerate(placed_rooms):
        for j in range(i+1, len(placed_rooms)):
            b = placed_rooms[j]
            ax, ay, aw, ah = a['pos']
            bx, by, bw, bh = b['pos']
            info = None
            if ax + aw == bx:      # b a droite de a
                lo, hi = max(ay, by), min(ay+ah, by+bh)
                if hi - lo >= 2: info = ('vertical', 'right', bx, lo, hi)
            elif bx + bw == ax:    # b a gauche de a
                lo, hi = max(ay, by), min(ay+ah, by+bh)
                if hi - lo >= 2: info = ('vertical', 'left', ax, lo, hi)
            elif ay + ah == by:    # b sous a
                lo, hi = max(ax, bx), min(ax+aw, bx+bw)
                if hi - lo >= 2: info = ('horizontal', 'bottom', by, lo, hi)
            elif by + bh == ay:    # b au-dessus de a
                lo, hi = max(ax, bx), min(ax+aw, bx+bw)
                if hi - lo >= 2: info = ('horizontal', 'top', ay, lo, hi)
            if info is None:
                continue
            a_allows = [x[0] for x in adjacencies.get(a['name'], [])]
            b_allows = [x[0] for x in adjacencies.get(b['name'], [])]
            legal = (b['name'] in a_allows) or (a['name'] in b_allows)
            overlap = info[4] - info[3]
            adj_pairs.append({'i': i, 'j': j, 'info': info,
                              'legal': legal, 'overlap': overlap})

    def make_door(a, b, info):
        orientation, side, fixed, lo, hi = info
        door_size = 2
        # Centrer la porte sur le segment partage (au lieu d'aleatoire)
        center = (lo + hi) // 2
        pos_along = max(lo, min(hi - door_size, center - door_size // 2))
        if orientation == 'vertical':
            door_pos = (fixed, pos_along)
        else:
            door_pos = (pos_along, fixed)
        return {'pos': door_pos, 'size': door_size,
                'orientation': orientation, 'side': side,
                'connected_rooms': (a['unique_id'], b['unique_id'])}

    placed_doors = []
    door_between = set()  # paires (i, j) deja reliees

    # --- 2. Poser les portes sur les adjacences LEGALES ---
    for p in adj_pairs:
        if p['legal']:
            a, b = placed_rooms[p['i']], placed_rooms[p['j']]
            placed_doors.append(make_door(a, b, p['info']))
            door_between.add((p['i'], p['j']))

    # --- 3. Connexite : chaque piece doit etre atteignable ---
    # On construit le graphe des portes puis on relie les composantes.
    def connected_components():
        parent = list(range(len(placed_rooms)))
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        def union(x, y):
            parent[find(x)] = find(y)
        for (i, j) in door_between:
            union(i, j)
        comps = {}
        for k in range(len(placed_rooms)):
            comps.setdefault(find(k), []).append(k)
        return list(comps.values())

    # Tant qu'il y a plusieurs composantes, on ajoute une porte "de secours"
    # entre deux composantes qui partagent un bord (meme si non legale).
    for _ in range(len(placed_rooms)):
        comps = connected_components()
        if len(comps) <= 1:
            break
        # Trouver une paire adjacente reliant deux composantes differentes
        comp_of = {}
        for ci, comp in enumerate(comps):
            for k in comp:
                comp_of[k] = ci
        best = None
        for p in adj_pairs:
            if (p['i'], p['j']) in door_between:
                continue
            if comp_of[p['i']] != comp_of[p['j']]:
                # Prioriser la plus grande zone de contact
                if best is None or p['overlap'] > best['overlap']:
                    best = p
        if best is None:
            break  # aucune adjacence physique pour relier -> on abandonne
        a, b = placed_rooms[best['i']], placed_rooms[best['j']]
        placed_doors.append(make_door(a, b, best['info']))
        door_between.add((best['i'], best['j']))

    # --- 4. Filet final : toute piece encore sans porte en recoit une ---
    has_door = set()
    for (i, j) in door_between:
        has_door.add(i); has_door.add(j)
    for k in range(len(placed_rooms)):
        if k in has_door:
            continue
        # Chercher n'importe quelle adjacence physique pour cette piece
        cand = None
        for p in adj_pairs:
            if p['i'] == k or p['j'] == k:
                if cand is None or p['overlap'] > cand['overlap']:
                    cand = p
        if cand is not None:
            a, b = placed_rooms[cand['i']], placed_rooms[cand['j']]
            placed_doors.append(make_door(a, b, cand['info']))
            door_between.add((cand['i'], cand['j']))

    # --- 5. Exterieur = tout ce qui est vide ---
    grid[grid == room_ids_map["vide"]] = room_ids_map["exterieur"]

    return grid, placed_doors


def place_rooms_compact(grid, pieces_to_place, place_rooms_fn,
                        room_ids_map, adjacencies,
                        place_door_fn=None):
    """Pipeline complet : placement initial -> compaction -> recentrage
    -> reconstruction grille + portes."""
    grid_h, grid_w = grid.shape

    # 1. Placement initial avec l'algo existant
    grid_init = np.zeros_like(grid)
    _, placed, _ = place_rooms_fn(grid_init, pieces_to_place)
    if not placed:
        return grid, [], []

    # 2. Compaction
    placed = compact_rooms(placed, grid_w, grid_h)

    # 3. Recentrage
    placed = recenter_rooms(placed, grid_w, grid_h)

    # 4. Reconstruction
    new_grid, doors = rebuild_grid_and_doors(
        placed, grid_w, grid_h, room_ids_map, place_door_fn, adjacencies)

    return new_grid, placed, doors


print("[Cellule 3 OK] Placement compact pret.")

## Cellule 3b — Placement v3 : graphe de bulles, portes en arbre, entrée, surface
Placement par défaut (`CFG['placement'] = 'v3'`). Le v2 reste disponible pour l'ablation.

In [ ]:
# =====================================================================
# CELLULE 3b - PLACEMENT v3 : GRAPHE DE BULLES + PORTES EN ARBRE
# =====================================================================
# Defauts mesures sur le placement v2 (300 programmes) :
#   1,9 porte par chambre, 1,6 par douche (pieces de passage),
#   17 % de chambres sans acces direct a une circulation,
#   10 % de WC ouvrant sur sejour/cuisine, compacite 0,66.
#
# v3 :
#   1. GRAPHE DE BULLES explicite construit a partir du programme
#      (sejour -> couloir -> chambres -> salle d'eau, suites parentales,
#      dressing et balcon rattaches a une chambre...). Le placement suit
#      ce graphe : chaque piece est posee contre SON parent.
#   2. POSITION : parmi toutes les positions valides le long du parent,
#      on retient la plus proche du centre de l'ensemble (compacite),
#      au lieu de 10 tirages aleatoires.
#   3. TAILLES variables dans des plages realistes, mises a l'echelle
#      si le prompt donne une surface ; couloir dimensionne selon le
#      nombre de pieces qu'il dessert.
#   4. PORTES EN ARBRE : arbre couvrant de cout minimal sur les contacts
#      (arete du graphe de bulles 0 < circulation 1 < autorisee 2 <
#      neutre 10 < interdite 100). Chaque piece privee a en general UNE
#      porte. Boucles autorisees seulement en zone jour (sejour, cuisine,
#      balcon, couloir).
#   5. PORTE D'ENTREE sur un mur exterieur du couloir (sinon du sejour).
# =====================================================================

import math

CIRCULATION = {"salon", "couloir"}
PRIVATE = {"chambre", "douche", "toilettes", "bureau", "dressing"}   # a ne pas traverser
# Annexes : acceptables via leur piece principale (suite parentale,
# WC dans la salle d'eau)
ANNEX_OF = {"dressing": "chambre", "douche": "chambre", "balcon": "chambre",
            "toilettes": "douche"}
DAY_ZONE = {"salon", "cuisine", "balcon", "couloir"}
FORBIDDEN = {frozenset(p) for p in [("toilettes", "cuisine"), ("chambre", "chambre"),
                                    ("douche", "cuisine"), ("chambre", "cuisine"),
                                    ("toilettes", "bureau"), ("toilettes", "chambre")]}

# Plages de tailles (cellules de 30 cm) : (largeur min, max), (hauteur min, max)
SIZE_RANGES = {
    "salon":     ((13, 16), (14, 18)),   # 16-26 m2
    "chambre":   ((9, 11), (10, 13)),    #  8-13 m2
    "cuisine":   ((8, 11), (9, 11)),     #  6-11 m2
    "douche":    ((6, 7), (7, 9)),       #  4-6 m2
    "toilettes": ((3, 4), (5, 6)),       #  1,4-2,2 m2
    "bureau":    ((7, 9), (8, 10)),
    "balcon":    ((5, 7), (9, 12)),
    "dressing":  ((4, 5), (5, 7)),
}
CELL_M2 = 0.09


def is_forbidden(na, nb, has_couloir):
    if frozenset((na, nb)) in FORBIDDEN:
        return True
    # WC sur le sejour : tolere seulement sans couloir (studio)
    return has_couloir and {na, nb} == {"toilettes", "salon"}


def is_legal(na, nb):
    return nb in [x[0] for x in room_adjacencies.get(na, [])] or \
           na in [x[0] for x in room_adjacencies.get(nb, [])]


# ---------------------------------------------------------------------
# 1. Graphe de bulles, parametre par une TOPOLOGIE de 6 entiers
# ---------------------------------------------------------------------
# Le graphe est un arbre de distribution dont la seule liberte est le
# RATTACHEMENT des pieces annexes. On le decrit par 6 entiers, generes par
# le cGAN au meme titre que les comptes de pieces :
TOPO_NAMES = ["sdb_suite",       # salles d'eau privatives (attachees a une chambre)
              "wc_dans_sdb",     # WC integres a la salle d'eau commune
              "dressing_suite",  # dressings attenants a une chambre
              "balcon_chambre",  # balcons desservis par une chambre
              "cuisine_fermee",  # cuisines desservies par le couloir (pas le sejour)
              "bureau_sejour"]   # bureaux ouverts sur le sejour


def clip_topology(pieces, topo):
    """Ramene une topologie dans le domaine realisable pour `pieces`."""
    n = lambda k: pieces.get(k, 0)
    has_c = n("couloir") > 0
    t = {k: max(0, int(topo.get(k, 0))) for k in TOPO_NAMES}
    # une salle d'eau commune est toujours conservee
    t["sdb_suite"] = min(t["sdb_suite"], max(0, n("douche") - 1), n("chambre"))
    t["wc_dans_sdb"] = min(t["wc_dans_sdb"], n("toilettes")) if n("douche") > 0 else 0
    t["dressing_suite"] = min(t["dressing_suite"], n("dressing")) if n("chambre") > 0 else 0
    t["balcon_chambre"] = min(t["balcon_chambre"], n("balcon"), n("chambre"))
    t["cuisine_fermee"] = min(t["cuisine_fermee"], n("cuisine")) if has_c else 0
    t["bureau_sejour"] = min(t["bureau_sejour"], n("bureau")) if has_c else n("bureau")
    return t


def default_topology(pieces):
    """Regles fixes du placement v3 (utilisees quand aucune topologie
    n'est fournie, et comme reference dans l'ablation)."""
    n = lambda k: pieces.get(k, 0)
    has_c = n("couloir") > 0
    return clip_topology(pieces, {
        "sdb_suite": n("douche") - 1, "wc_dans_sdb": 0 if has_c else n("toilettes"),
        "dressing_suite": n("dressing"), "balcon_chambre": max(0, n("balcon") - 1),
        "cuisine_fermee": 0, "bureau_sejour": 0 if has_c else n("bureau")})


def bubble_graph(pieces, topo=None):
    """Liste ordonnee (parents avant enfants) de dicts
    {'uid', 'name', 'parent'} decrivant l'arbre de distribution."""
    topo = default_topology(pieces) if topo is None else clip_topology(pieces, topo)
    nodes, count = [], {}

    def add(name, parent):
        k = count.get(name, 0); count[name] = k + 1
        uid = f"{name}_{k}"
        nodes.append({"uid": uid, "name": name, "parent": parent})
        return uid

    salon = add("salon", None)
    for _ in range(pieces.get("salon", 1) - 1):
        add("salon", salon)
    hub = add("couloir", salon) if pieces.get("couloir", 0) > 0 else None
    for _ in range(pieces.get("couloir", 0) - 1):
        add("couloir", hub)
    night = hub or salon
    for i in range(pieces.get("cuisine", 0)):
        add("cuisine", hub if i < topo["cuisine_fermee"] else salon)
    chambres = [add("chambre", night) for _ in range(pieces.get("chambre", 0))]
    n_shared = pieces.get("douche", 0) - topo["sdb_suite"]
    douches = []
    for i in range(pieces.get("douche", 0)):
        parent = night if i < n_shared else chambres[(i - n_shared) % len(chambres)]
        douches.append(add("douche", parent))
    for i in range(pieces.get("toilettes", 0)):
        add("toilettes", douches[0] if i < topo["wc_dans_sdb"] else night)
    for i in range(pieces.get("bureau", 0)):
        add("bureau", salon if i < topo["bureau_sejour"] else night)
    for i in range(pieces.get("dressing", 0)):
        add("dressing", chambres[i % len(chambres)] if i < topo["dressing_suite"] else night)
    for i in range(pieces.get("balcon", 0)):
        add("balcon", chambres[i % len(chambres)] if i < topo["balcon_chambre"] else salon)
    return nodes


# ---------------------------------------------------------------------
# 2. Tailles
# ---------------------------------------------------------------------
def expected_area_m2(pieces):
    a = 0.0
    for n, k in pieces.items():
        if n in SIZE_RANGES:
            (w0, w1), (h0, h1) = SIZE_RANGES[n]
            a += k * (w0 + w1) / 2 * (h0 + h1) / 2 * CELL_M2
    n_served = pieces.get("chambre", 0) + pieces.get("bureau", 0)
    a += pieces.get("couloir", 0) * 3.5 * (4 + 11 * math.ceil(n_served / 2)) * CELL_M2
    return a


def area_scale(pieces, target_m2):
    if not target_m2:
        return 1.0
    return float(np.clip(math.sqrt(target_m2 / expected_area_m2(pieces)), 0.75, 1.5))


def room_dims(name, pieces, scale):
    if name == "couloir":
        n_served = pieces.get("chambre", 0) + pieces.get("bureau", 0)
        length = 4 + 11 * math.ceil(max(1, n_served) / 2)
        return random.choice([3, 4]), max(8, round(length * scale))
    (w0, w1), (h0, h1) = SIZE_RANGES[name]
    return (max(3, round(random.randint(w0, w1) * scale)),
            max(3, round(random.randint(h0, h1) * scale)))


# ---------------------------------------------------------------------
# 3. Placement guide par le graphe
# ---------------------------------------------------------------------
def _candidate_positions(par, w, h, min_overlap=3):
    """Toutes les positions (x, y) ou un rectangle w x h touche `par`
    sur au moins min_overlap cellules."""
    px, py, pw, ph = par['pos']
    out = []
    for x in range(px - w + min_overlap, px + pw - min_overlap + 1):
        out.append((x, py - h)); out.append((x, py + ph))
    for y in range(py - h + min_overlap, py + ph - min_overlap + 1):
        out.append((px - w, y)); out.append((px + pw, y))
    return out


def place_rooms_graph(pieces, grid_size, scale=1.0, topo=None):
    nodes = bubble_graph(pieces, topo)
    placed, by_uid = [], {}
    for node in nodes:
        name = node["name"]
        w, h = room_dims(name, pieces, scale)
        orients = [(w, h), (h, w)] if w != h else [(w, h)]
        random.shuffle(orients)
        if node["parent"] is None:
            x, y = (grid_size - w) // 2, (grid_size - h) // 2
            r = {'id': room_ids[name], 'name': name, 'unique_id': node["uid"],
                 'pos': (x, y, w, h), 'parent': None, 'graph_parent': None}
            placed.append(r); by_uid[node["uid"]] = r
            continue
        # Parent du graphe d'abord, puis pieces autorisees, puis les autres
        par0 = by_uid.get(node["parent"])
        others = [p for p in placed if p is not par0]
        others.sort(key=lambda p: (not is_legal(name, p['name']), random.random()))
        parents = ([par0] if par0 else []) + others
        cx = np.mean([p['pos'][0] + p['pos'][2] / 2 for p in placed])
        cy = np.mean([p['pos'][1] + p['pos'][3] / 2 for p in placed])
        best = None
        for par in parents:
            if is_forbidden(name, par['name'], pieces.get("couloir", 0) > 0):
                continue
            for (ww, hh) in orients:
                for (x, y) in _candidate_positions(par, ww, hh):
                    pos = (x, y, ww, hh)
                    if not can_place_at(None, pos, placed, grid_size, grid_size):
                        continue
                    d = (x + ww / 2 - cx) ** 2 + (y + hh / 2 - cy) ** 2
                    if best is None or d < best[0]:
                        best = (d, pos, par)
            if best is not None:
                break          # on reste sur le premier parent qui convient
        if best is None:
            continue           # piece impossible a placer
        r = {'id': room_ids[name], 'name': name, 'unique_id': node["uid"],
             'pos': best[1], 'parent': best[2]['unique_id'],
             'graph_parent': node["parent"]}
        placed.append(r); by_uid[node["uid"]] = r
    return placed


# ---------------------------------------------------------------------
# 4. Portes en arbre + porte d'entree
# ---------------------------------------------------------------------
def find_contacts(placed, min_len=2):
    out = []
    for i, a in enumerate(placed):
        ax, ay, aw, ah = a['pos']
        for j in range(i + 1, len(placed)):
            bx, by, bw, bh = placed[j]['pos']
            info = None
            if ax + aw == bx:
                lo, hi = max(ay, by), min(ay + ah, by + bh)
                if hi - lo >= min_len: info = ('vertical', 'right', bx, lo, hi)
            elif bx + bw == ax:
                lo, hi = max(ay, by), min(ay + ah, by + bh)
                if hi - lo >= min_len: info = ('vertical', 'left', ax, lo, hi)
            elif ay + ah == by:
                lo, hi = max(ax, bx), min(ax + aw, bx + bw)
                if hi - lo >= min_len: info = ('horizontal', 'bottom', by, lo, hi)
            elif by + bh == ay:
                lo, hi = max(ax, bx), min(ax + aw, bx + bw)
                if hi - lo >= min_len: info = ('horizontal', 'top', ay, lo, hi)
            if info:
                out.append((i, j, info))
    return out


def _door(info, a_uid, b_uid, kind="interieure"):
    orientation, side, fixed, lo, hi = info
    along = max(lo, min(hi - 2, (lo + hi) // 2 - 1))
    pos = (fixed, along) if orientation == 'vertical' else (along, fixed)
    return {'pos': pos, 'size': 2, 'orientation': orientation, 'side': side,
            'connected_rooms': (a_uid, b_uid), 'kind': kind}


def door_cost(a, b, has_couloir):
    if b.get('graph_parent') == a['unique_id'] or a.get('graph_parent') == b['unique_id']:
        base = 0
    elif is_forbidden(a['name'], b['name'], has_couloir):
        return 100
    elif is_legal(a['name'], b['name']):
        base = 1 if (a['name'] in CIRCULATION or b['name'] in CIRCULATION) else 2
    else:
        base = 10
    if is_forbidden(a['name'], b['name'], has_couloir):
        base = 100
    return base


def _loop_ok(a, b):
    """Une porte de boucle a ne doit pas contredire le rattachement voulu :
    une cuisine fermee (rattachee au couloir) ne s'ouvre pas sur le sejour."""
    gp = a.get('graph_parent')
    return gp is None or not gp.startswith("couloir") or a['name'] != "cuisine"


def doors_tree(placed, grid, has_couloir):
    contacts = find_contacts(placed)
    parent = list(range(len(placed)))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x

    doors, used = [], set()
    for i, j, info in sorted(contacts, key=lambda c: (door_cost(placed[c[0]], placed[c[1]], has_couloir),
                                                      -(c[2][4] - c[2][3]))):
        if find(i) != find(j):
            parent[find(i)] = find(j)
            doors.append(_door(info, placed[i]['unique_id'], placed[j]['unique_id']))
            used.add((i, j))
    # Boucles autorisees en zone jour (sejour ouvert sur cuisine, balcon...)
    for i, j, info in contacts:
        a, b = placed[i], placed[j]
        # (pas de boucle pour une piece que le graphe rattache ailleurs,
        #  ex. cuisine fermee desservie par le couloir)
        if (i, j) not in used and a['name'] in DAY_ZONE and b['name'] in DAY_ZONE \
                and is_legal(a['name'], b['name']) \
                and _loop_ok(a, b) and _loop_ok(b, a):
            doors.append(_door(info, a['unique_id'], b['unique_id']))
    # Porte d'entree : couloir sinon sejour, sur le plus long mur exterieur
    H, W = grid.shape
    ext = room_ids["exterieur"]
    for target in ("couloir", "salon"):
        cands = [r for r in placed if r['name'] == target]
        best = None
        for r in cands:
            x, y, w, h = r['pos']
            sides = [('horizontal', 'top', y, [(c, y - 1) for c in range(x, x + w)]),
                     ('horizontal', 'bottom', y + h, [(c, y + h) for c in range(x, x + w)]),
                     ('vertical', 'left', x, [(x - 1, rr) for rr in range(y, y + h)]),
                     ('vertical', 'right', x + w, [(x + w, rr) for rr in range(y, y + h)])]
            for orient, side, fixed, cells in sides:
                run, start = 0, 0
                for k, (c, rr) in enumerate(cells):
                    outside = not (0 <= c < W and 0 <= rr < H) or grid[rr, c] == ext
                    run = run + 1 if outside else 0
                    if run and (best is None or run > best[0]):
                        lo = (x if orient == 'horizontal' else y) + k - run + 1
                        best = (run, (orient, side, fixed, lo, lo + run), r)
        if best and best[0] >= 3:
            doors.append(_door(best[1], best[2]['unique_id'], "exterieur", kind="entree"))
            break
    return doors


# ---------------------------------------------------------------------
# Fenetres : sur les murs exterieurs, centrees sur le plus long troncon
# libre, avec un trumeau d'au moins 30 cm aux angles. Largeurs en cellules
# de 30 cm : sejour 1,80 m (et une seconde baie si possible), chambre,
# bureau et cuisine 1,20 m, salle de bain et WC 60 cm (chassis haut).
# ---------------------------------------------------------------------
WINDOW_SPEC = {"salon": (6, 4), "chambre": (4, 3), "bureau": (4, 3), "cuisine": (4, 3),
               "douche": (2, 2), "toilettes": (2, 2)}
LIT_ROOMS = {"salon", "chambre", "bureau", "cuisine"}


def windows_for(placed, doors):
    """Liste des fenetres {'pos', 'size', 'orientation', 'room', 'high'}."""
    occ = set()
    for r in placed:
        x, y, w, h = r['pos']
        occ.update((c, rr) for c in range(x, x + w) for rr in range(y, y + h))
    entry = next((d for d in doors if d.get('kind') == 'entree'), None)
    out = []
    for r in placed:
        spec = WINDOW_SPEC.get(r['name'])
        if not spec:
            continue
        x, y, w, h = r['pos']
        sides = [('horizontal', y, x, w, lambda k: (x + k, y - 1)),
                 ('horizontal', y + h, x, w, lambda k: (x + k, y + h)),
                 ('vertical', x, y, h, lambda k: (x - 1, y + k)),
                 ('vertical', x + w, y, h, lambda k: (x + w, y + k))]
        runs = []
        for orient, fixed, start, length, cell in sides:
            lo = None
            for k in range(length + 1):
                free = k < length and cell(k) not in occ
                if free and entry and entry['orientation'] == orient \
                        and entry['connected_rooms'][0] == r['unique_id']:
                    ef, ea = (entry['pos'] if orient == 'vertical' else entry['pos'][::-1])
                    if ef == fixed and ea - 1 <= start + k < ea + entry['size'] + 1:
                        free = False
                if free and lo is None:
                    lo = k
                if not free and lo is not None:
                    runs.append((orient, fixed, start + lo, k - lo)); lo = None
        runs.sort(key=lambda t: (-t[3], t[2]))
        want, mini = spec
        n = 0
        for orient, fixed, lo, length in runs:
            size = min(want, length - 2)
            if size < mini:
                break
            along = lo + (length - size) // 2
            out.append({'pos': (fixed, along) if orient == 'vertical' else (along, fixed),
                        'size': size, 'orientation': orient, 'room': r['unique_id'],
                        'high': r['name'] not in LIT_ROOMS})
            n += 1
            if n >= (2 if r['name'] == 'salon' else 1):
                break
    return out


def daylight_ratio(placed, doors, windows=None):
    """Part des pieces de vie qui recoivent le jour (fenetre ou porte-fenetre)."""
    windows = windows_for(placed, doors) if windows is None else windows
    lit = {w['room'] for w in windows}
    names = {r['unique_id']: r['name'] for r in placed}
    for d in doors:
        a, b = d['connected_rooms']
        if names.get(a) == "balcon": lit.add(b)
        if names.get(b) == "balcon": lit.add(a)
    rooms = [r for r in placed if r['name'] in LIT_ROOMS]
    return sum(r['unique_id'] in lit for r in rooms) / len(rooms) if rooms else 1.0


def place_rooms_v3(grid, pieces, target_m2=None, compact=None, topo=None):
    # La compaction n'apporte rien en v3 (positions deja choisies au plus
    # pres du centre) et coute ~6x le temps : desactivee par defaut.
    compact = CFG.get("v3_compact", False) if compact is None else compact
    gs = grid.shape[0]
    scale = area_scale(pieces, target_m2)
    placed = place_rooms_graph(pieces, gs, scale, topo)
    if not placed:
        return grid, [], []
    if compact:
        placed = compact_rooms(placed, gs, gs)
    placed = recenter_rooms(placed, gs, gs)
    new_grid = np.full((gs, gs), room_ids["exterieur"], dtype=int)
    for r in placed:
        x, y, w, h = r['pos']
        new_grid[y:y + h, x:x + w] = r['id']
    doors = doors_tree(placed, new_grid, pieces.get("couloir", 0) > 0)
    return new_grid, placed, doors


def grid_size_for(pieces, target_m2=None):
    """Grille assez grande pour le programme (au moins 60 cellules)."""
    area = max(expected_area_m2(pieces), target_m2 or 0) / CELL_M2
    return int(max(60, math.ceil(2.2 * math.sqrt(area))))


# --- Test ---
random.seed(0)
for prog in [{"salon": 1, "cuisine": 1, "douche": 1, "toilettes": 1},
             {"salon": 1, "cuisine": 1, "chambre": 1, "douche": 1, "toilettes": 1, "balcon": 1},
             {"salon": 1, "cuisine": 1, "chambre": 3, "couloir": 1, "douche": 2,
              "toilettes": 1, "bureau": 1, "dressing": 1}]:
    print([(n['name'], n['parent']) for n in bubble_graph(prog)])
print("\n[Cellule 3b OK] Placement v3 pret.")

## Cellule 4 — Parser de prompt (texte → contraintes)

In [ ]:
# =====================================================================
# CELLULE 4 - PARSER DE PROMPT
# =====================================================================
"""
prompt_parser.py -- Etage 1 du prompter : texte -> vecteur de contraintes.

Transforme un prompt libre ou structure en un vecteur de dimension 9
(ordre = GENERATIVE_ROOMS). Chaque entree vaut :
  - un entier >= 0 : nombre impose de cette piece
  - -1            : "libre", le cGAN decidera

Exemples :
  "T3 avec balcon"                  -> chambre=2, balcon=1, salon=1, cuisine=1, reste=-1
  "grand appart, 2 douches, bureau" -> douche=2, bureau=1, reste=-1
  "studio"                          -> chambre=0, salon=1, cuisine=1
"""

import re
import numpy as np

GENERATIVE_ROOMS = ["salon", "chambre", "douche", "toilettes", "couloir",
                    "bureau", "cuisine", "balcon", "dressing"]
# Vecteur complet genere par le cGAN : 9 comptes de pieces + 6 variables
# de topologie du graphe de bulles (TOPO_NAMES, cellule 3b)
VECTOR_NAMES = GENERATIVE_ROOMS + TOPO_NAMES
ROOM_INDEX = {name: i for i, name in enumerate(VECTOR_NAMES)}

# Expressions qui fixent la TOPOLOGIE : (motif, variable, valeur, pieces a
# garantir). valeur "n" = autant que de pieces concernees (au moins 1).
TOPO_PATTERNS = [
    (r"suite parentale|salle d.eau privative|douche privative|sdb privative|chambre avec (?:sa )?(?:salle de bains?|salle d.eau|douche)",
     "sdb_suite", 1, {"douche": 2, "chambre": 1}),
    (r"(?:wc|toilettes?) s[ée]par[ée]s?|(?:wc|toilettes?) ind[ée]pendante?s?",
     "wc_dans_sdb", 0, {}),
    (r"(?:wc|toilettes?) dans la (?:salle de bains?|salle d.eau)",
     "wc_dans_sdb", 1, {"toilettes": 1, "douche": 1}),
    (r"dressing (?:dans|attenant|de) la chambre|chambre avec dressing",
     "dressing_suite", 1, {"dressing": 1, "chambre": 1}),
    (r"balcon (?:dans|depuis|de) la chambre|chambre avec balcon",
     "balcon_chambre", 1, {"balcon": 1, "chambre": 1}),
    (r"cuisine (?:ferm[ée]e|ind[ée]pendante|s[ée]par[ée]e)",
     "cuisine_fermee", 1, {"cuisine": 1, "couloir": 1}),
    (r"cuisine (?:ouverte|am[ée]ricaine)",
     "cuisine_fermee", 0, {}),
    (r"bureau (?:ouvert sur|dans) (?:le )?(?:s[ée]jour|salon)|coin bureau",
     "bureau_sejour", 1, {"bureau": 1}),
    (r"bureau (?:ferm[ée]|ind[ée]pendant)",
     "bureau_sejour", 0, {"bureau": 1}),
]

# --- Dictionnaire de synonymes -> nom canonique ---
SYNONYMS = {
    "salon": ["salon", "sejour", "séjour", "living", "piece a vivre",
              "pièce à vivre", "salle de vie"],
    "chambre": ["chambre", "chambres", "chb", "bedroom"],
    "douche": ["douche", "douches", "salle d'eau", "salle d eau",
               "salle de bain", "sdb", "salle de bains", "bain"],
    "toilettes": ["toilettes", "toilette", "wc", "w.c.", "waters",
                  "cabinet"],
    "couloir": ["couloir", "couloirs", "corridor", "degagement",
                "dégagement", "hall"],
    "bureau": ["bureau", "bureaux", "office", "espace de travail",
               "coin bureau"],
    "cuisine": ["cuisine", "cuisines", "kitchen", "cuisine equipee",
                "cuisine équipée"],
    "balcon": ["balcon", "balcons", "terrasse", "loggia", "balconnet"],
    "dressing": ["dressing", "penderie", "walk-in", "walk in",
                 "garde-robe", "garde robe"],
}

# Nombres ecrits en toutes lettres (fr)
WORD_NUMBERS = {
    "un": 1, "une": 1, "deux": 2, "trois": 3, "quatre": 4, "cinq": 5,
    "six": 6, "sept": 7, "huit": 8, "zero": 0, "zéro": 0,
    "aucun": 0, "aucune": 0, "pas de": 0, "sans": 0,
}

# Typologies -> nombre de chambres
TYPO_TO_CHAMBRES = {
    "studio": 0, "t1": 0, "f1": 0,
    "t2": 1, "f2": 1,
    "t3": 2, "f3": 2,
    "t4": 3, "f4": 3,
    "t5": 4, "f5": 4,
    "t6": 5, "f6": 5,
}


def _normalize(text):
    """Minuscule, sans accents parasites pour le matching."""
    text = text.lower().strip()
    # On garde les accents dans les synonymes mais on tolere sans
    text = text.replace("’", "'")
    return text


def _find_number_before_after(text, keyword):
    """Cherche un nombre juste avant ou apres un mot-cle.
    Ex: '2 douches' -> 2 ; 'douche x3' -> 3 ; 'deux chambres' -> 2."""
    # Motifs : "<nombre> <keyword>" ou "<keyword> <nombre>"
    # nombre = chiffre ou mot
    num_pattern = r"(\d+|" + "|".join(WORD_NUMBERS.keys()) + r")"

    # Avant : "2 chambres", "deux chambres"
    # \b : evite que "un" matche la fin d'un mot ("brun chambre")
    m = re.search(r"\b" + num_pattern + r"\s+" + re.escape(keyword) + r"\b",
                  text)
    if m:
        return _to_int(m.group(1))
    # Apres : "chambre x2", "chambres : 2". Le separateur est obligatoire :
    # sinon "cuisine 2 chambres" donnerait cuisine=2.
    m = re.search(re.escape(keyword) + r"\s*[:x×]\s*" + num_pattern, text)
    if m:
        return _to_int(m.group(1))
    return None


def _to_int(token):
    if token.isdigit():
        return int(token)
    return WORD_NUMBERS.get(token, None)


def parse_prompt(prompt, default_free=True):
    """Transforme un prompt en vecteur de contraintes (np.array dim 9).

    Args:
      prompt : le texte (libre ou structure)
      default_free : si True, les pieces non mentionnees = -1 (libre).
                     si False, elles = 0 (absentes).

    Returns:
      np.array de taille 9, dtype float. -1 = libre.
    """
    text = _normalize(prompt)
    fill = -1.0 if default_free else 0.0
    vec = np.full(len(VECTOR_NAMES), fill, dtype=np.float32)

    # 1. Typologie (T1..T5, studio) -> fixe le nb de chambres
    for typo, n_ch in TYPO_TO_CHAMBRES.items():
        if re.search(r"\b" + re.escape(typo) + r"\b", text):
            vec[ROOM_INDEX["chambre"]] = n_ch
            # Un logement a toujours salon + cuisine
            vec[ROOM_INDEX["salon"]] = 1
            vec[ROOM_INDEX["cuisine"]] = 1
            break

    # 2. Pieces explicitement mentionnees (avec ou sans nombre)
    for canonical, syns in SYNONYMS.items():
        idx = ROOM_INDEX[canonical]
        found_syn = None
        # On teste chaque synonyme, y compris sa variante plurielle
        for syn in sorted(syns, key=len, reverse=True):  # + longs d'abord
            variants = [syn]
            # Variante plurielle simple : "salle d'eau" -> "salles d'eau"
            if " " in syn:
                first, rest = syn.split(" ", 1)
                variants.append(first + "s " + rest)
            else:
                variants.append(syn + "s")
            for var in variants:
                if var in text:
                    found_syn = var
                    break
            if found_syn:
                break

        if found_syn:
            n = _find_number_before_after(text, found_syn)
            if n is not None:
                vec[idx] = n
            else:
                neg = re.search(r"(sans|pas de|aucune?|zero|zéro)\s+"
                                + re.escape(found_syn), text)
                vec[idx] = 0 if neg else (max(1, int(vec[idx]))
                                          if vec[idx] > 0 else 1)

    # 3. Topologie explicite ("suite parentale", "cuisine fermee"...)
    #    + pieces que cette topologie suppose (si elles sont libres)
    for pattern, var, value, needs in TOPO_PATTERNS:
        if re.search(pattern, text):
            vec[ROOM_INDEX[var]] = value
            for room, k in needs.items():
                if vec[ROOM_INDEX[room]] == -1:   # un nombre explicite prime
                    vec[ROOM_INDEX[room]] = k

    # 4. Coherence minimale : salon + cuisine par defaut presents
    #    (sauf si explicitement mis a 0)
    if vec[ROOM_INDEX["salon"]] == -1:
        vec[ROOM_INDEX["salon"]] = 1
    if vec[ROOM_INDEX["cuisine"]] == -1:
        vec[ROOM_INDEX["cuisine"]] = 1

    return vec


def parse_footprint(prompt):
    """Surface / emprise demandee. Retourne {'surface_m2', 'dims'} ou {}.
      "appartement de 70 m2", "T3 de 65m²", "plan 10x12 m", "9 x 11 metres"
    Les dimensions sans unite ("plan 100x100" du memoire) sont ignorees."""
    text = _normalize(prompt).replace(",", ".")
    num = r"(\d+(?:\.\d+)?)"
    m = re.search(num + r"\s*[x×]\s*" + num + r"\s*(?:m\b|metres?\b|mètres?\b)", text)
    if m:
        a, b = float(m.group(1)), float(m.group(2))
        return {"surface_m2": a * b, "dims": (a, b)}
    m = re.search(num + r"\s*(?:m2|m²|m\^2|metres? carres?|mètres? carrés?)", text)
    if m:
        return {"surface_m2": float(m.group(1)), "dims": None}
    return {}


def vec_to_readable(vec):
    """Affiche un vecteur de contraintes de facon lisible."""
    parts = []
    for i, name in enumerate(VECTOR_NAMES):
        v = vec[i]
        if v == -1:
            parts.append(f"{name}=libre")
        else:
            parts.append(f"{name}={int(v)}")
    return "  ".join(parts)


# --- Tests rapides ---

TEST_PROMPTS = [
    "T3 avec balcon",
    "T4 avec 2 douches et un bureau",
    "studio",
    "appart 5 chambres sans balcon",
    "Un salon , deux chambre , une douche ",
    "grand appartement avec cuisine et 2 chambres",
    "chambres : 3, wc x2, pas de balcon",
    "T3 de 65 m² avec balcon",
    "plan 9x11 m avec 2 chambres",
    "T4 avec suite parentale et cuisine fermée",
    "T2 avec wc séparés et coin bureau",
]
for p in TEST_PROMPTS:
    fp = parse_footprint(p)
    print(f"{p!r:50s} -> {vec_to_readable(parse_prompt(p))}"
          + (f"  | surface {fp['surface_m2']:.0f} m2" if fp else ""))

print("\n[Cellule 4 OK] Parser pret.")

## Cellule 5 — Métriques de qualité d'un plan

In [ ]:
# =====================================================================
# CELLULE 5 - METRIQUES DE QUALITE D'UN PLAN
# =====================================================================
# Utilisees a la fois pour filtrer le dataset (cellule 7) et pour
# l'evaluation (cellule 10). Toutes renvoient un float dans [0, 1].
#
#   connectivity_from_salon : part des pieces atteignables depuis le salon
#                             par les portes (BFS) - metrique du memoire v1
#   is_connected            : 1 si le graphe des portes est connexe
#   legal_door_ratio        : part des portes reliant deux pieces dont
#                             l'adjacence est autorisee (room_adjacencies)
#   evaluate_adjacency_compliance (cellule 1) : part des pieces ayant au
#                             moins un voisin autorise via une porte
#   compactness             : surface des pieces / surface de la boite
#                             englobante (1 = aucun vide interieur)
#   forbidden_doors         : nombre de portes interdites (WC->cuisine,
#                             chambre->chambre...)
#   direct_access           : part des pieces atteignables depuis l'entree
#                             sans traverser une piece privee (exceptions :
#                             suite parentale - dressing, salle d'eau,
#                             balcon via la chambre - et WC dans la salle d'eau)
#   doors_per_private       : nombre moyen de portes des pieces privees
#   ecart_surface           : |surface obtenue - demandee| / demandee
# =====================================================================

from collections import deque, Counter
import heapq


def _door_graph(placed, doors):
    ids = [r['unique_id'] for r in placed]
    graph = {u: set() for u in ids}
    for d in doors:
        a, b = d.get('connected_rooms', (None, None))
        if a in graph and b in graph:
            graph[a].add(b); graph[b].add(a)
    return graph


def is_connected(placed, doors):
    if not placed:
        return False
    graph = _door_graph(placed, doors)
    start = placed[0]['unique_id']
    seen, todo = {start}, deque([start])
    while todo:
        for n in graph[todo.popleft()]:
            if n not in seen:
                seen.add(n); todo.append(n)
    return len(seen) == len(placed)


def connectivity_from_salon(placed, doors):
    if not placed:
        return 0.0
    graph = _door_graph(placed, doors)
    starts = [r['unique_id'] for r in placed if r['name'] == 'salon']
    start = starts[0] if starts else placed[0]['unique_id']
    seen, todo = {start}, deque([start])
    while todo:
        for n in graph[todo.popleft()]:
            if n not in seen:
                seen.add(n); todo.append(n)
    return len(seen) / len(placed)


def legal_door_ratio(placed, doors, adj=None):
    adj = room_adjacencies if adj is None else adj
    doors = [d for d in doors if d.get('kind') != 'entree']
    if not doors:
        return 0.0
    names = {r['unique_id']: r['name'] for r in placed}
    legal = 0
    for d in doors:
        a, b = d['connected_rooms']
        na, nb = names.get(a), names.get(b)
        if nb in [x[0] for x in adj.get(na, [])] or \
           na in [x[0] for x in adj.get(nb, [])]:
            legal += 1
    return legal / len(doors)


def compactness(placed):
    if not placed:
        return 0.0
    x0 = min(r['pos'][0] for r in placed); y0 = min(r['pos'][1] for r in placed)
    x1 = max(r['pos'][0] + r['pos'][2] for r in placed)
    y1 = max(r['pos'][1] + r['pos'][3] for r in placed)
    area = sum(r['pos'][2] * r['pos'][3] for r in placed)
    return area / max(1, (x1 - x0) * (y1 - y0))


def forbidden_doors(placed, doors, has_couloir):
    names = {r['unique_id']: r['name'] for r in placed}
    n = 0
    for d in doors:
        a, b = d['connected_rooms']
        if a in names and b in names and is_forbidden(names[a], names[b], has_couloir):
            n += 1
    return n


def _entrance_root(placed, doors):
    for d in doors:
        if d.get('kind') == 'entree':
            return d['connected_rooms'][0]
    salons = [r['unique_id'] for r in placed if r['name'] == 'salon']
    return salons[0] if salons else (placed[0]['unique_id'] if placed else None)


def direct_access(placed, doors):
    if len(placed) < 2:
        return 1.0
    graph = _door_graph(placed, doors)
    names = {r['unique_id']: r['name'] for r in placed}
    root = _entrance_root(placed, doors)
    dist = {root: 0}; heap = [(0, root)]
    while heap:                               # Dijkstra : cout = pieces privees traversees
        d, u = heapq.heappop(heap)
        if d > dist.get(u, 1e9):
            continue
        step = 1 if (u != root and names[u] in PRIVATE) else 0
        for v in graph[u]:
            if d + step < dist.get(v, 1e9):
                dist[v] = d + step; heapq.heappush(heap, (d + step, v))
    ok = 0
    for u in names:
        if u == root:
            continue
        if dist.get(u, 1e9) == 0:
            ok += 1
        elif names[u] in ANNEX_OF and dist.get(u, 1e9) == 1 and any(
                names[c] == ANNEX_OF[names[u]] and dist.get(c, 1e9) == 0 for c in graph[u]):
            ok += 1
    return ok / (len(names) - 1)


def doors_per_private(placed, doors):
    deg = Counter()
    for d in doors:
        if d.get('kind') == 'entree':
            continue
        a, b = d['connected_rooms']; deg[a] += 1; deg[b] += 1
    priv = [deg[r['unique_id']] for r in placed if r['name'] in PRIVATE]
    return float(np.mean(priv)) if priv else 0.0


def realized_topology(placed, doors):
    """Topologie effectivement obtenue, lue sur les portes du plan."""
    names = {r['unique_id']: r['name'] for r in placed}
    nb = {u: set() for u in names}
    for d in doors:
        a, b = d['connected_rooms']
        if a in nb and b in nb:
            nb[a].add(names[b]); nb[b].add(names[a])
    circ = {"salon", "couloir"}
    t = dict.fromkeys(TOPO_NAMES, 0)
    for u, n in names.items():
        if n == "douche" and "chambre" in nb[u] and not nb[u] & circ:
            t["sdb_suite"] += 1
        elif n == "toilettes" and "douche" in nb[u] and not nb[u] & circ:
            t["wc_dans_sdb"] += 1
        elif n == "dressing" and "chambre" in nb[u] and not nb[u] & circ:
            t["dressing_suite"] += 1
        elif n == "balcon" and "chambre" in nb[u] and "salon" not in nb[u]:
            t["balcon_chambre"] += 1
        elif n == "cuisine" and "couloir" in nb[u] and "salon" not in nb[u]:
            t["cuisine_fermee"] += 1
        elif n == "bureau" and "salon" in nb[u]:
            t["bureau_sejour"] += 1
    return t


def realized_vector(placed, doors):
    """Comptes de pieces placees + topologie obtenue (meme ordre que
    VECTOR_NAMES) : sert a mesurer le respect du prompt sur le plan final."""
    counts = [sum(r['name'] == n for r in placed) for n in GENERATIVE_ROOMS]
    t = realized_topology(placed, doors)
    return np.array(counts + [t[k] for k in TOPO_NAMES])


def surface_m2(placed):
    return sum(r['pos'][2] * r['pos'][3] for r in placed) * CELL_M2


def plan_metrics(pieces, placed, doors, target_m2=None):
    """Toutes les metriques d'un plan, sous forme de dict."""
    requested = sum(pieces.values())
    has_c = pieces.get("couloir", 0) > 0
    s = surface_m2(placed)
    return {
        "sans_porte_interdite": float(forbidden_doors(placed, doors, has_c) == 0),
        "acces_direct":  direct_access(placed, doors),
        "portes_piece_privee": doors_per_private(placed, doors),
        "entree":        float(any(d.get('kind') == 'entree' for d in doors)),
        "surface_m2":    s,
        "ecart_surface": abs(s - target_m2) / target_m2 if target_m2 else float("nan"),
        "placement":    min(1.0, len(placed) / max(1, requested)),
        "connectivite": connectivity_from_salon(placed, doors),
        "connexe":      float(is_connected(placed, doors)),
        "portes_legales": legal_door_ratio(placed, doors),
        "adjacence":    evaluate_adjacency_compliance(placed, doors),
        "compacite":    compactness(placed),
        "eclairement":  daylight_ratio(placed, doors),
    }


print("[Cellule 5 OK] Metriques pretes.")

## Cellule 6 — Briques du cGAN
Dataset conditionnel, G(z, c), D(v, c), perte de contraintes, entraînement, génération.

In [ ]:
# =====================================================================
# CELLULE 6 - BRIQUES DU cGAN
# =====================================================================
#   6.1 Dataset conditionnel (masquage aleatoire -> condition 18 dims)
#   6.2 Architectures G(z, c) et D(v, c) + perte de contraintes
#   6.3 Boucle d'entrainement centralisee
#   6.4 Generation : prompt -> vecteur -> programme (pieces + graphe) -> plan
#
# Le vecteur genere a 15 dimensions : 9 comptes de pieces (GENERATIVE_ROOMS)
# + 6 variables de topologie du graphe de bulles (TOPO_NAMES).
# =====================================================================

import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset

VECTOR_DIM = len(VECTOR_NAMES)       # 9 pieces + 6 topologie = 15
COND_DIM = 2 * VECTOR_DIM            # valeurs + masque
LATENT_DIM = CFG["latent_dim"]


# ---------------------------------------------------------------------
# 6.1 Dataset conditionnel
# ---------------------------------------------------------------------
class ConditionalRoomDataset(Dataset):
    """Pour chaque vrai vecteur v, fabrique a la volee une condition c en
    masquant aleatoirement certaines entrees. Le cGAN apprend a generer v
    a partir de (z, c) en respectant les entrees non masquees.

    condition = concat [values, mask] (dim 18)
      values : valeur normalisee (0 si masque)
      mask   : 1 si impose, 0 si libre
    """
    def __init__(self, vectors_tensor, max_vec, noise_std=0.02,
                 mask_min=0.3, mask_max=0.9):
        self.v_norm = vectors_tensor / max_vec   # normalise dans [0,1]
        self.noise_std = noise_std
        self.mask_min = mask_min
        self.mask_max = mask_max
        self.dim = vectors_tensor.shape[1]

    def __len__(self):
        return len(self.v_norm)

    def __getitem__(self, idx):
        v = self.v_norm[idx].clone()
        # Cible : vrai vecteur + petit bruit (lisse la distribution discrete)
        target = (v + torch.randn_like(v) * self.noise_std).clamp(0, 1)
        # Condition : proportion aleatoire d'entrees masquees (libres)
        mask_ratio = np.random.uniform(self.mask_min, self.mask_max)
        mask = (torch.rand(self.dim) > mask_ratio).float()   # 1 = impose
        values = v * mask
        return target, torch.cat([values, mask], dim=0)


def make_condition_from_constraints(constraint_vec, max_vec):
    """Vecteur de contraintes (parser, -1 = libre) -> condition (18,).
    Les valeurs au-dela du max du dataset sont ecretees a 1 : le reseau
    n'a jamais vu plus que max_vec (le SNAP retablira la valeur demandee)."""
    mv = max_vec if torch.is_tensor(max_vec) else torch.tensor(max_vec, dtype=torch.float32)
    cv = torch.as_tensor(constraint_vec, dtype=torch.float32, device=mv.device)
    mask = (cv >= 0).float()
    values = torch.where(cv >= 0, cv / mv, torch.zeros_like(cv)).clamp(0, 1)
    return torch.cat([values, mask], dim=-1)


# ---------------------------------------------------------------------
# 6.2 Architectures
# ---------------------------------------------------------------------
class ConditionalGenerator(nn.Module):
    """G(z, c) -> vecteur de pieces normalise [0,1] (MLP)."""
    def __init__(self, latent_dim=LATENT_DIM, cond_dim=COND_DIM,
                 out_dim=VECTOR_DIM):
        super().__init__()
        self.z_dim = latent_dim
        self.cond_dim = cond_dim
        self.model = nn.Sequential(
            nn.Linear(latent_dim + cond_dim, 128), nn.LeakyReLU(0.2), nn.BatchNorm1d(128),
            nn.Linear(128, 128), nn.LeakyReLU(0.2), nn.BatchNorm1d(128),
            nn.Linear(128, 64), nn.LeakyReLU(0.2),
            nn.Linear(64, out_dim), nn.Sigmoid()
        )

    def forward(self, z, c):
        return self.model(torch.cat([z, c], dim=1))


class ConditionalDiscriminator(nn.Module):
    """D(v, c) -> probabilite que (v, c) soit un vrai couple."""
    def __init__(self, vector_dim=VECTOR_DIM, cond_dim=COND_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vector_dim + cond_dim, 128), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(128, 128), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.LeakyReLU(0.2),
            nn.Linear(64, 1), nn.Sigmoid()
        )

    def forward(self, v, c):
        return self.net(torch.cat([v, c], dim=1))


def constraint_violation_loss(generated, condition):
    """MSE entre la sortie de G et les valeurs imposees, calculee
    UNIQUEMENT sur les entrees imposees (mask = 1)."""
    d = generated.shape[1]
    values, mask = condition[:, :d], condition[:, d:]
    diff = (generated - values) * mask
    return (diff ** 2).sum() / (mask.sum() + 1e-8)


def count_params(m):
    return sum(p.numel() for p in m.parameters())


# ---------------------------------------------------------------------
# 6.3 Entrainement centralise
# ---------------------------------------------------------------------
def train_cgan(G, D, dataloader, device, epochs=200, lr=2e-4,
               lambda_constraint=15.0, checkpoint_dir=None,
               checkpoint_every=25, log_every=10, verbose=True):
    """A chaque batch :
      1. D apprend : (v_reel, c) -> reel ; (G(z,c), c) -> faux
      2. G apprend : tromper D + lambda * respect des contraintes
    lambda_constraint est LE bouton de reglage (cf. ablation cellule 11).
    """
    opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    bce = nn.BCELoss()
    hist = {"d_loss": [], "g_loss": [], "g_adv": [], "g_constraint": [], "d_acc": []}
    t0 = time.time()

    for epoch in range(epochs):
        G.train(); D.train()
        ep_d, ep_g, ep_adv, ep_con = [], [], [], []
        d_correct, d_total = 0, 0
        for target, cond in dataloader:
            target, cond = target.to(device), cond.to(device)
            b = target.size(0)
            real_lbl = torch.full((b, 1), 0.9, device=device)   # label smoothing
            fake_lbl = torch.full((b, 1), 0.1, device=device)

            # ---------- D ----------
            D.zero_grad()
            out_real = D(target, cond)
            fake = G(torch.randn(b, G.z_dim, device=device), cond).detach()
            out_fake = D(fake, cond)
            d_loss = bce(out_real, real_lbl) + bce(out_fake, fake_lbl)
            d_loss.backward(); opt_D.step()
            d_correct += ((out_real > 0.5).sum() + (out_fake < 0.5).sum()).item()
            d_total += 2 * b

            # ---------- G ----------
            G.zero_grad()
            fake = G(torch.randn(b, G.z_dim, device=device), cond)
            g_adv = bce(D(fake, cond), real_lbl)
            g_con = constraint_violation_loss(fake, cond)
            g_loss = g_adv + lambda_constraint * g_con
            g_loss.backward(); opt_G.step()

            ep_d.append(d_loss.item()); ep_g.append(g_loss.item())
            ep_adv.append(g_adv.item()); ep_con.append(g_con.item())

        hist["d_loss"].append(float(np.mean(ep_d)))
        hist["g_loss"].append(float(np.mean(ep_g)))
        hist["g_adv"].append(float(np.mean(ep_adv)))
        hist["g_constraint"].append(float(np.mean(ep_con)))
        hist["d_acc"].append(d_correct / d_total)

        if verbose and ((epoch + 1) % log_every == 0 or epoch == 0):
            print(f"Ep {epoch+1:>3}/{epochs} | D={hist['d_loss'][-1]:.3f} | "
                  f"G_adv={hist['g_adv'][-1]:.3f} | "
                  f"G_constraint={hist['g_constraint'][-1]:.4f} | "
                  f"D_acc={hist['d_acc'][-1]*100:4.1f}% | {time.time()-t0:.0f}s")

        if checkpoint_dir and (epoch + 1) % checkpoint_every == 0:
            torch.save({"G": G.state_dict(), "D": D.state_dict(),
                        "epoch": epoch + 1, "hist": hist},
                       os.path.join(checkpoint_dir, f"cgan_ep{epoch+1:03d}.pt"))

    hist["train_time_s"] = time.time() - t0
    if verbose:
        print(f"\nTermine en {hist['train_time_s']/60:.1f} min")
    return hist


def plot_history(hist, save_path=None, title_prefix=""):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
    ax[0].plot(hist["d_loss"], label="D"); ax[0].plot(hist["g_adv"], label="G (adversariale)")
    ax[0].set_title(f"{title_prefix}Pertes"); ax[0].set_xlabel("époque"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(hist["g_constraint"], color="crimson")
    ax[1].set_yscale("log"); ax[1].set_title(f"{title_prefix}Violation des contraintes (MSE)")
    ax[1].set_xlabel("époque"); ax[1].grid(alpha=.3)
    ax[2].plot([a * 100 for a in hist["d_acc"]], color="purple")
    ax[2].axhline(50, ls="--", color="gray")
    ax[2].set_title(f"{title_prefix}Exactitude de D (%)"); ax[2].set_xlabel("époque"); ax[2].grid(alpha=.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


# ---------------------------------------------------------------------
# 6.4 Generation
# ---------------------------------------------------------------------
@torch.no_grad()
def generate_counts(G, constraints, max_vec, snap=True):
    """constraints : array (n, 15) avec -1 = libre.
    Retourne (raw, final) : entiers (n, 15) avant / apres SNAP."""
    device = next(G.parameters()).device
    mv = (max_vec if torch.is_tensor(max_vec) else torch.tensor(max_vec)).float().to(device)
    cons = torch.as_tensor(np.asarray(constraints), dtype=torch.float32, device=device)
    cond = make_condition_from_constraints(cons, mv)
    G.eval()
    out = G(torch.randn(len(cons), G.z_dim, device=device), cond)
    raw = torch.round(out * mv).long().cpu().numpy()
    final = raw.copy()
    if snap:
        c = np.asarray(constraints)
        final[c >= 0] = c[c >= 0].astype(int)
    return raw, final


def counts_to_pieces(vec, room_names=GENERATIVE_ROOMS):
    """9 premieres entrees -> dict de pieces + regles de coherence minimales
    (salon et cuisine toujours presents, couloir des 2 chambres)."""
    pieces = {name: int(vec[i]) for i, name in enumerate(room_names) if vec[i] > 0}
    pieces.setdefault("salon", 1)
    pieces.setdefault("cuisine", 1)
    if pieces.get("chambre", 0) >= 2:
        pieces.setdefault("couloir", 1)
    return pieces


def vector_to_program(vec):
    """Vecteur (15,) -> (pieces, topologie realisable)."""
    pieces = counts_to_pieces(vec)
    n = len(GENERATIVE_ROOMS)
    topo = clip_topology(pieces, {k: int(vec[n + i]) for i, k in enumerate(TOPO_NAMES)})
    return pieces, topo


def program_to_vector(pieces, topo=None):
    topo = default_topology(pieces) if topo is None else topo
    return np.array([pieces.get(k, 0) for k in GENERATIVE_ROOMS] +
                    [topo[k] for k in TOPO_NAMES], dtype=np.float32)


def pieces_to_vector(pieces):          # compatibilite
    return program_to_vector(pieces)


def _layout_score(pieces, placed, doors):
    """Ordre lexicographique : toutes les pieces placees > plan connexe >
    aucune porte interdite > acces sans traversee > pieces de vie eclairees >
    portes legales > compacite."""
    has_c = pieces.get("couloir", 0) > 0
    return (min(len(placed), sum(pieces.values())), is_connected(placed, doors),
            -forbidden_doors(placed, doors, has_c), direct_access(placed, doors),
            daylight_ratio(placed, doors), legal_door_ratio(placed, doors), compactness(placed))


def layout_pieces(pieces, grid_size=None, tries=CFG["layout_tries"],
                  placement=CFG["placement"], target_m2=None, topo=None):
    """Placement (v2 glouton ou v3 graphe de bulles) + compaction + portes.
    Le placement etant stochastique, on fait jusqu'a `tries` essais et on
    s'arrete au premier plan complet, connexe, sans porte interdite ni
    piece traversee, pieces de vie eclairees et a >= 90 % de portes legales ; sinon le meilleur."""
    if grid_size is None:
        grid_size = CFG["grid_size"] if placement == "v2" else grid_size_for(pieces, target_m2)
    best, best_score = None, None
    for _ in range(tries):
        grid = np.zeros((grid_size, grid_size), dtype=int)
        if placement == "v2":
            out = place_rooms_compact(grid, pieces, place_rooms, room_ids, room_adjacencies)
        else:
            out = place_rooms_v3(grid, pieces, target_m2=target_m2, topo=topo)
        score = _layout_score(pieces, out[1], out[2])
        if best is None or score > best_score:
            best, best_score = out, score
        if score[0] == sum(pieces.values()) and score[1] and score[2] == 0 \
                and score[3] == 1.0 and score[4] == 1.0 and score[5] >= 0.90:
            break
    return best


def plan_from_prompt(prompt, G, max_vec, n_variants=1, snap=True, seed=None,
                     show=True, save_dir=None, verbose=True):
    """prompt -> parser -> cGAN -> SNAP -> pieces -> placement -> rendu."""
    if seed is not None:
        set_seed(seed)
    t0 = time.perf_counter()
    constraint = parse_prompt(prompt)
    target_m2 = parse_footprint(prompt).get("surface_m2")
    t_parse = time.perf_counter() - t0
    if verbose:
        imposed = {VECTOR_NAMES[i]: int(v) for i, v in enumerate(constraint) if v >= 0}
        print(f"Prompt   : \"{prompt}\"")
        print(f"Contraintes imposees : {imposed}")

    t0 = time.perf_counter()
    raw, final = generate_counts(G, np.repeat(constraint[None], n_variants, 0), max_vec, snap)
    t_gen = (time.perf_counter() - t0) / n_variants

    results = []
    for k in range(n_variants):
        pieces, topo = vector_to_program(final[k])
        t0 = time.perf_counter()
        grid, placed, doors = layout_pieces(pieces, target_m2=target_m2, topo=topo)
        t_layout = time.perf_counter() - t0

        title = f"« {prompt} »" + (f"  (variante {k+1})" if n_variants > 1 else "")
        save_path = None
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            safe = "".join(c if c.isalnum() else "_" for c in prompt)[:30]
            save_path = os.path.join(save_dir, f"{safe}_{k+1}.png")
        if show or save_path:
            render_architectural_plan(grid, placed, doors, room_ids, id_to_room,
                                      title=title, save_path=save_path, show=show)
        results.append({"pieces": pieces, "topo": topo, "grid": grid, "placed": placed,
                        "doors": doors, "constraint": constraint, "raw": raw[k],
                        "metrics": plan_metrics(pieces, placed, doors, target_m2),
                        "times": {"parse": t_parse, "cgan": t_gen, "layout": t_layout}})
        if verbose:
            print(f"  Variante {k+1}: {pieces}")
            print(f"             graphe: { {k_: v for k_, v in topo.items() if v} }")
    return results


print("[Cellule 6 OK] Briques cGAN pretes.")

## Cellule 7 — Dataset de programmes + découpage apprentissage / test
Mis en cache sur Drive : la génération (~20 min) n'a lieu qu'une fois.

In [ ]:
# =====================================================================
# CELLULE 7 - DATASET DE PROGRAMMES + DECOUPAGE APPRENTISSAGE / TEST
# =====================================================================
# Un "programme" = vecteur de 15 entiers : nombre de pieces par type (9)
# + topologie du graphe de bulles (6, cf. TOPO_NAMES). La topologie est
# tiree avec des preferences realistes qui DEPENDENT de la typologie
# (WC dans la salle d'eau surtout sans couloir, suite parentale quand il
# y a 2 salles d'eau...) : c'est cette dependance que le cGAN doit apprendre.
# On tire des programmes selon une distribution de typologies (T1..T5)
# et on ne garde que ceux que le pipeline de placement sait realiser :
#   toutes les pieces placees + plan connexe + >= 90 % de portes legales.
# Le fichier est mis en cache sur Drive (generation ~20 min).
# Le nom du fichier indique le placement utilise pour valider les
# programmes ; "r2" = regles d'adjacence revisees (WC sur sejour sans couloir).
# =====================================================================

from collections import Counter
from torch.utils.data import DataLoader

DATASET_PATH = os.path.join(BASE_DIR, f"dataset_programs_{CFG['placement']}g_r2_{CFG['n_vectors']}.pt")

TYPOLOGIES = {
    "T1": {"weight": 0.10, "chambres": [0, 1]},
    "T2": {"weight": 0.20, "chambres": [1]},
    "T3": {"weight": 0.35, "chambres": [2]},
    "T4": {"weight": 0.25, "chambres": [3]},
    "T5": {"weight": 0.10, "chambres": [4]},
}


def sample_candidate(typo=None):
    if typo is None:
        tn = list(TYPOLOGIES); tw = [TYPOLOGIES[t]["weight"] for t in tn]
        typo = random.choices(tn, weights=tw, k=1)[0]
    nc = random.choice(TYPOLOGIES[typo]["chambres"])
    p = {"salon": 1, "cuisine": 1, "chambre": nc,
         "toilettes": 1 if random.random() < 0.9 else 2,
         "douche": 1 if nc <= 2 else random.choice([1, 2])}
    if nc >= 2: p["couloir"] = 1
    if nc >= 1 and random.random() < 0.4: p["dressing"] = 1
    if random.random() < 0.3: p["bureau"] = 1
    if random.random() < 0.5: p["balcon"] = 1
    return {k: v for k, v in p.items() if v > 0}


def sample_topology(p):
    n = lambda k: p.get(k, 0)
    has_c = n("couloir") > 0
    bern = lambda k, q: sum(random.random() < q for _ in range(k))
    return clip_topology(p, {
        "sdb_suite":      bern(max(0, n("douche") - 1), 0.85),
        "wc_dans_sdb":    bern(n("toilettes"), 0.15 if has_c else 0.7),
        "dressing_suite": bern(n("dressing"), 0.8),
        "balcon_chambre": bern(n("balcon"), 0.25),
        "cuisine_fermee": bern(n("cuisine"), 0.35),
        "bureau_sejour":  bern(n("bureau"), 0.3),
    })


def validate(pieces, topo=None, tries=3):
    total = sum(pieces.values())
    for _ in range(tries):
        _, placed, doors = layout_pieces(pieces, tries=1, topo=topo)
        if len(placed) >= total and is_connected(placed, doors) and \
           legal_door_ratio(placed, doors) >= 0.90:
            return True
    return False


if os.path.exists(DATASET_PATH):
    data = torch.load(DATASET_PATH, weights_only=False)
    vectors_tensor = data["vectors"]; max_per_room = data["max_per_room"]
    RESULTS["dataset_gen"] = data.get("gen_stats", {})
    print(f"Dataset charge: {tuple(vectors_tensor.shape)}")
else:
    print(f"Generation du dataset ({CFG['n_vectors']} programmes)...")
    set_seed(SEED)
    vecs, n_tried, t0 = [], 0, time.time()
    while len(vecs) < CFG["n_vectors"]:
        p = sample_candidate(); t = sample_topology(p); n_tried += 1
        if validate(p, t):
            vecs.append(program_to_vector(p, t))
            if len(vecs) % max(1, CFG["n_vectors"] // 10) == 0:
                print(f"  [{len(vecs)}/{CFG['n_vectors']}] {time.time()-t0:.0f}s")
    vectors_tensor = torch.from_numpy(np.stack(vecs))
    max_per_room = np.maximum(vectors_tensor.numpy().max(0), 1).tolist()   # >= 1
    gen_stats = {"tried": n_tried, "kept": len(vecs),
                 "acceptance": len(vecs) / n_tried, "time_s": time.time() - t0}
    RESULTS["dataset_gen"] = gen_stats
    torch.save({"vectors": vectors_tensor, "room_names": VECTOR_NAMES,
                "max_per_room": max_per_room, "gen_stats": gen_stats}, DATASET_PATH)
    print(f"Sauvegarde: {DATASET_PATH} | taux d'acceptation {gen_stats['acceptance']:.1%}")

MAX_VEC = torch.tensor(max_per_room, dtype=torch.float32)

# --- Decoupage apprentissage / test (graine fixe) ---------------------
perm = torch.randperm(len(vectors_tensor), generator=torch.Generator().manual_seed(SEED))
n_test = int(CFG["test_ratio"] * len(vectors_tensor))
test_vectors = vectors_tensor[perm[:n_test]]
train_vectors = vectors_tensor[perm[n_test:]]

# Support = ensemble des programmes realisables observes. Un vecteur
# genere est dit "plausible" s'il appartient a ce support.
SUPPORT = Counter(tuple(int(x) for x in v) for v in vectors_tensor.numpy())

# --- Statistiques descriptives (pour l'article) ----------------------
n_ch = Counter(int(v[GENERATIVE_ROOMS.index("chambre")]) for v in vectors_tensor)
RESULTS["dataset"] = {
    "n": len(vectors_tensor), "n_train": len(train_vectors), "n_test": len(test_vectors),
    "n_unique": len(SUPPORT), "max_per_room": dict(zip(VECTOR_NAMES, max_per_room)),
    "n_unique_pieces": len({k[:len(GENERATIVE_ROOMS)] for k in SUPPORT}),
    "topo_mean": {k: float(vectors_tensor[:, len(GENERATIVE_ROOMS) + i].mean())
                  for i, k in enumerate(TOPO_NAMES)},
    "mean_rooms": float(vectors_tensor[:, :len(GENERATIVE_ROOMS)].sum(1).mean()),
    "chambres_hist": {str(k): v for k, v in sorted(n_ch.items())},
}
print(f"Train {len(train_vectors)} | Test {len(test_vectors)} | "
      f"programmes distincts {len(SUPPORT)} (dont {RESULTS['dataset']['n_unique_pieces']} "
      f"sans la topologie) | pieces/plan {RESULTS['dataset']['mean_rooms']:.2f}")
print(f"max par entree: {dict(zip(VECTOR_NAMES, max_per_room))}")

# --- DataLoader conditionnel ----------------------------------------
NUM_WORKERS = 2 if IN_COLAB else 0
cond_dataset = ConditionalRoomDataset(train_vectors, MAX_VEC)
cond_dataloader = DataLoader(cond_dataset, batch_size=CFG["batch_size"], shuffle=True,
                             drop_last=True, num_workers=NUM_WORKERS)
print(f"DataLoader conditionnel: {len(cond_dataloader)} batches de {CFG['batch_size']}")
print("\n[Cellule 7 OK]")

## Cellule 8 — Entraînement centralisé du cGAN (référence)

In [ ]:
# =====================================================================
# CELLULE 8 - ENTRAINEMENT CENTRALISE DU cGAN (reference)
# =====================================================================
# Toutes les donnees sont reunies au meme endroit : c'est la borne
# "ideale" a laquelle on comparera le Split Learning (cellule 13).
# Surveiller G_constraint : il doit descendre (G respecte les prompts).
# =====================================================================

set_seed(SEED)
G_cond = ConditionalGenerator().to(device)
D_cond = ConditionalDiscriminator().to(device)
print(f"G: {count_params(G_cond):,} params | D: {count_params(D_cond):,} params")

hist = train_cgan(G_cond, D_cond, cond_dataloader, device,
                  epochs=CFG["epochs"], lr=CFG["lr"], lambda_constraint=CFG["lambda"],
                  checkpoint_dir=MODEL_DIR, checkpoint_every=25, log_every=10)

torch.save({"G": G_cond.state_dict(), "D": D_cond.state_dict(),
            "latent_dim": LATENT_DIM, "cond_dim": COND_DIM,
            "max_per_room": max_per_room, "room_names": GENERATIVE_ROOMS},
           os.path.join(MODEL_DIR, "cgan_final.pt"))

RESULTS["central_training"] = {
    "params_G": count_params(G_cond), "params_D": count_params(D_cond),
    "epochs": CFG["epochs"], "time_s": hist["train_time_s"],
    "final": {k: hist[k][-1] for k in ("d_loss", "g_adv", "g_constraint", "d_acc")},
}
json.dump(hist, open(os.path.join(EXPORT_DIR, "hist_central.json"), "w"))
plot_history(hist, save_path=os.path.join(EXPORT_DIR, "fig_training_central.png"))
print("\n[Cellule 8 OK]")

## Cellule 9 — Le prompter `plan()`

In [ ]:
# =====================================================================
# CELLULE 9 - LE PROMPTER : plan() en une ligne
# =====================================================================
#   plan("T3 avec balcon")
#   plan("T4 avec 2 douches et un bureau", n=3)
#   plan("studio", show=False, save_dir=EXPORT_DIR)
# =====================================================================
import matplotlib.pyplot as plt


def plan(prompt, n=1, snap=True, seed=None, show=True, save_dir=None, G=None):
    return plan_from_prompt(prompt, G if G is not None else G_cond, MAX_VEC,
                            n_variants=n, snap=snap, seed=seed, show=show,
                            save_dir=save_dir)


# --- Figure d'exemples pour l'article (2 x 2) ------------------------
EXAMPLE_PROMPTS = ["studio avec balcon", "T3 avec balcon",
                   "T4 avec 2 douches et un bureau", "appart 5 chambres sans balcon"]
fig, axes = plt.subplots(2, 2, figsize=(14, 14), facecolor="white")
for ax, p in zip(axes.flat, EXAMPLE_PROMPTS):
    r = plan(p, seed=SEED, show=False)[0]
    render_architectural_plan(r["grid"], r["placed"], r["doors"], room_ids, id_to_room,
                              title=p, ax=ax, title_size=11)
plt.tight_layout()
plt.savefig(os.path.join(EXPORT_DIR, "fig_exemples_plans.png"), dpi=200,
            bbox_inches="tight", facecolor="white")
plt.show()

# Un exemple interactif
_ = plan("T4 avec 2 douches et un bureau", n=2, seed=7)
print("\n[Cellule 9 OK]")

## Cellule 10 — Évaluation quantitative
Méthode proposée (cGAN + snap) contre trois références : cGAN sans snap, tirage aléatoire, prior empirique (recherche dans le dataset).

In [ ]:
# =====================================================================
# CELLULE 10 - EVALUATION QUANTITATIVE
# =====================================================================
# Methodes comparees (meme parser, meme placement, meme rendu) :
#   cgan_snap : methode proposee (cGAN + SNAP des contraintes explicites)
#   cgan_brut : cGAN seul, sans SNAP (mesure ce que le reseau a appris)
#   aleatoire : pieces libres tirees uniformement dans [0, max]
#   prior     : on tire un programme du dataset d'apprentissage
#               compatible avec les contraintes (baseline par recherche)
#   cgan_snap_1essai : methode proposee avec un seul essai de placement
#               (ablation du best-of-k de layout_pieces)
#   cgan_snap_graphe_regles : pieces du cGAN mais graphe de bulles par
#               regles fixes (ablation de la prediction du graphe)
#   cgan_snap_placement_v2 : placement glouton v2 (ablation du placement)
#
# Deux niveaux d'evaluation :
#   A. Vecteurs (rapide) : conditions tirees du jeu de TEST
#        respect_brut : part des contraintes satisfaites par la sortie brute
#        plausibilite : programme final dans le support des programmes
#                       realisables
#   B. Plans (complet) : prompts de reference -> plans complets
#        + placement, connectivite, portes legales, adjacence, compacite,
#          diversite, temps
# =====================================================================

import io
import statistics as st

rng_eval = np.random.default_rng(SEED)
train_np = train_vectors.numpy().astype(int)
max_np = np.array(max_per_room, dtype=int)


def method_cgan(G, snap):
    def f(cons):
        return generate_counts(G, cons, MAX_VEC, snap=snap)
    return f


def method_random(cons):
    cons = np.asarray(cons)
    out = rng_eval.integers(0, max_np + 1, size=cons.shape)
    out[cons >= 0] = cons[cons >= 0].astype(int)
    return out.copy(), out


def method_prior(cons):
    cons = np.asarray(cons); out = []
    for c in cons:
        m = c >= 0
        dist = np.abs(train_np[:, m] - c[m]).sum(1) if m.any() else np.zeros(len(train_np))
        cand = np.flatnonzero(dist == dist.min())
        v = train_np[rng_eval.choice(cand)].copy()
        v[m] = c[m].astype(int)
        out.append(v)
    out = np.array(out)
    return out.copy(), out


def final_vector(vec, graph="cgan"):
    """Programme final (pieces + topologie realisable) sous forme de tuple.
    graph="regles" : topologie remplacee par les regles fixes."""
    pieces, topo = vector_to_program(vec)
    return tuple(int(x) for x in program_to_vector(pieces, topo if graph == "cgan" else None))


TOPO_IDX = np.arange(len(GENERATIVE_ROOMS), len(VECTOR_NAMES))


def respect_rate(counts, cons):
    """Part des entrees imposees exactement respectees."""
    m = cons >= 0
    return float((counts[m] == cons[m]).mean()) if m.any() else 1.0


def mean_ci(xs):
    xs = [float(x) for x in xs]
    m = st.mean(xs) if xs else float("nan")
    ci = 1.96 * st.stdev(xs) / np.sqrt(len(xs)) if len(xs) > 1 else 0.0
    return m, ci


def make_test_constraints(vectors, n, p_impose=0.5, seed=SEED):
    r = np.random.default_rng(seed)
    v = vectors.numpy()[r.integers(0, len(vectors), n)].copy()
    mask = r.random(v.shape) < p_impose
    v[~mask] = -1
    return v


# ---------------------------------------------------------------------
# A. Evaluation vectorielle sur le jeu de test
# ---------------------------------------------------------------------
def eval_vectors(gen_fn, cons):
    raw, final = gen_fn(cons)
    resp = [respect_rate(raw[i], cons[i]) for i in range(len(cons))]
    plaus = [final_vector(final[i]) in SUPPORT for i in range(len(cons))]
    return {"respect_brut": float(np.mean(resp)), "plausibilite": float(np.mean(plaus))}


TEST_CONS = make_test_constraints(test_vectors, CFG["eval_test_conds"])
METHODS = {"cgan_snap": method_cgan(G_cond, True), "cgan_brut": method_cgan(G_cond, False),
           "aleatoire": method_random, "prior": method_prior}

RESULTS["eval_vectors"] = {}
print("A. Evaluation vectorielle (conditions du jeu de test)")
for name, fn in METHODS.items():
    set_seed(SEED)
    RESULTS["eval_vectors"][name] = eval_vectors(fn, TEST_CONS)
    r = RESULTS["eval_vectors"][name]
    print(f"  {name:10s} respect brut {r['respect_brut']:.3f} | plausibilite {r['plausibilite']:.3f}")

# ---------------------------------------------------------------------
# B. Evaluation des plans complets sur les prompts de reference
# ---------------------------------------------------------------------
BENCH_PROMPTS = [
    "studio", "studio avec balcon", "T2", "T2 avec bureau", "T3", "T3 avec balcon",
    "T3 avec bureau et dressing", "T4", "T4 avec 2 douches et un bureau",
    "T5 avec balcon", "un salon, deux chambres, une douche",
    "appartement 3 chambres sans balcon", "grand appartement avec bureau",
    "appart 5 chambres sans balcon",          # hors distribution (max 4 chambres)
    "studio de 30 m2", "T3 de 65 m² avec balcon", "T4 de 95 m2",   # surface imposee
    # topologie imposee (graphe de bulles)
    "T4 avec suite parentale", "T3 avec cuisine fermée", "T2 avec wc séparés",
    "studio avec wc dans la salle de bain", "T3 avec chambre avec dressing et coin bureau",
]
TOPO_PROMPTS = BENCH_PROMPTS[-5:]
N_VAR = CFG["eval_variants"]
PLAN_KEYS = ["placement", "connectivite", "connexe", "portes_legales", "adjacence", "compacite",
             "sans_porte_interdite", "acces_direct", "portes_piece_privee", "entree"]


def eval_plans(gen_fn, prompts, n_var, time_render=False, tries=CFG["layout_tries"],
               placement=CFG["placement"], graph="cgan"):
    rows = []
    for p in prompts:
        t0 = time.perf_counter(); cons1 = parse_prompt(p)
        target = parse_footprint(p).get("surface_m2"); t_parse = time.perf_counter() - t0
        cons = np.repeat(cons1[None], n_var, 0)
        t0 = time.perf_counter(); raw, final = gen_fn(cons)
        t_gen = (time.perf_counter() - t0) / n_var
        finals, graphs = [], []
        topo_imposed = (cons1[TOPO_IDX] >= 0).any()
        for k in range(n_var):
            pieces, topo = vector_to_program(final[k])
            if graph != "cgan":
                topo = default_topology(pieces)
            t0 = time.perf_counter()
            grid, placed, doors = layout_pieces(pieces, tries=tries, placement=placement,
                                                target_m2=target, topo=topo)
            t_layout = time.perf_counter() - t0
            realized = realized_vector(placed, doors)
            row = {"prompt": p, "respect_brut": respect_rate(raw[k], cons1),
                   "respect_plan": respect_rate(realized, cons1),
                   # respect de la topologie imposee, lu sur les portes du plan
                   "respect_graphe": respect_rate(realized[TOPO_IDX], cons1[TOPO_IDX])
                                     if topo_imposed else float("nan"),
                   "plausibilite": float(final_vector(final[k], graph) in SUPPORT),
                   "t_parse": t_parse, "t_cgan": t_gen, "t_layout": t_layout,
                   "t_total": t_parse + t_gen + t_layout, "n_pieces": len(placed)}
            row.update(plan_metrics(pieces, placed, doors, target))
            if time_render and k == 0:
                t0 = time.perf_counter()
                render_architectural_plan(grid, placed, doors, room_ids, id_to_room, title=p,
                                          save_path=io.BytesIO(), show=False)   # PNG en memoire
                row["t_render"] = time.perf_counter() - t0
            rows.append(row); finals.append(final_vector(final[k], graph))
            graphs.append(tuple(realized_topology(placed, doors).values()))
        for r in rows[-n_var:]:
            r["diversite"] = len(set(finals)) / n_var
            r["diversite_graphe"] = len(set(graphs)) / n_var
    return rows


RESULTS["eval_plans"] = {}
PLAN_ROWS = {}
print(f"\nB. Evaluation des plans ({len(BENCH_PROMPTS)} prompts x {N_VAR} variantes)")
# + ablations du placement : un seul essai au lieu de best-of-k, et
#   placement v2 (glouton) au lieu de v3 (graphe de bulles)
PLAN_METHODS = dict(METHODS, cgan_snap_1essai=METHODS["cgan_snap"],
                    cgan_snap_graphe_regles=METHODS["cgan_snap"],
                    cgan_snap_placement_v2=METHODS["cgan_snap"])
for name, fn in PLAN_METHODS.items():
    set_seed(SEED); t0 = time.time()
    rows = eval_plans(fn, BENCH_PROMPTS, N_VAR, time_render=(name == "cgan_snap"),
                      tries=1 if name.endswith("1essai") else CFG["layout_tries"],
                      placement="v2" if name.endswith("_v2") else CFG["placement"],
                      graph="regles" if name.endswith("graphe_regles") else "cgan")
    PLAN_ROWS[name] = rows
    agg = {}
    for k in ["respect_brut", "respect_plan", "plausibilite", "diversite", "diversite_graphe",
              *PLAN_KEYS, "t_cgan", "t_layout", "t_total"]:
        m, ci = mean_ci([r[k] for r in rows]); agg[k] = {"mean": m, "ci95": ci}
    rg = [r["respect_graphe"] for r in rows if r["respect_graphe"] == r["respect_graphe"]]
    agg["respect_graphe"] = {"mean": float(np.mean(rg)), "ci95": mean_ci(rg)[1]} if rg else \
                            {"mean": float("nan"), "ci95": 0.0}
    tt = [r["t_total"] for r in rows]
    agg["t_total"].update({"min": min(tt), "max": max(tt), "median": float(np.median(tt))})
    if name == "cgan_snap":
        tr = [r["t_render"] for r in rows if "t_render" in r]
        agg["t_render"] = {"mean": float(np.mean(tr)), "ci95": 0.0}
    es = [r["ecart_surface"] for r in rows if r["ecart_surface"] == r["ecart_surface"]]
    if es:
        agg["ecart_surface"] = {"mean": float(np.mean(es)), "ci95": 0.0}
    RESULTS["eval_plans"][name] = agg
    print(f"  {name:22s} ({time.time()-t0:.0f}s) " + " | ".join(
        f"{k} {agg[k]['mean']:.3f}" for k in ["respect_plan", "respect_graphe", "plausibilite",
                                              "diversite", "diversite_graphe",
                                              "portes_legales", "sans_porte_interdite",
                                              "acces_direct", "portes_piece_privee",
                                              "compacite"]))

# Detail par prompt pour la methode proposee
RESULTS["eval_plans_per_prompt"] = {}
print("\nDetail cgan_snap par prompt:")
for p in BENCH_PROMPTS:
    rs = [r for r in PLAN_ROWS["cgan_snap"] if r["prompt"] == p]
    d = {k: float(np.mean([r[k] for r in rs])) for k in
         ["respect_brut", "respect_plan", "plausibilite", "diversite", "diversite_graphe", "n_pieces",
          "connectivite", "portes_legales", "t_total"]}
    RESULTS["eval_plans_per_prompt"][p] = d
    print(f"  {p:38s} pieces {d['n_pieces']:4.1f} | respect brut {d['respect_brut']:.2f} | "
          f"plan {d['respect_plan']:.2f} | plaus {d['plausibilite']:.2f} | "
          f"div {d['diversite']:.2f} | graphes {d['diversite_graphe']:.2f} | "
          f"{d['t_total']*1000:.0f} ms")

# Prompts qui imposent la topologie : graphe predit vs regles fixes
RESULTS["eval_topo_prompts"] = {}
print("\nPrompts topologiques : respect du graphe impose (lu sur les portes)")
for p in TOPO_PROMPTS:
    d = {m: float(np.mean([r["respect_graphe"] for r in PLAN_ROWS[m] if r["prompt"] == p]))
         for m in ("cgan_snap", "cgan_snap_graphe_regles")}
    RESULTS["eval_topo_prompts"][p] = d
    print(f"  {p:45s} cGAN {d['cgan_snap']:.2f} | regles {d['cgan_snap_graphe_regles']:.2f}")

json.dump(PLAN_ROWS, open(os.path.join(EXPORT_DIR, "eval_plans_rows.json"), "w"))
print("\n[Cellule 10 OK]")

## Cellule 11 — Ablation du poids λ

In [ ]:
# =====================================================================
# CELLULE 11 - ABLATION : poids lambda de la perte de contraintes
# =====================================================================
# lambda = 0 -> cGAN purement adversarial ; lambda grand -> G recopie la
# condition. On mesure le compromis respect / plausibilite / diversite.
# (L'ablation du SNAP est deja donnee par cgan_brut vs cgan_snap, cell. 10)
# =====================================================================


def vector_diversity(gen_fn, cons, n_samples=20):
    """Nombre moyen de programmes distincts (normalise par n_samples)
    generes pour une meme condition."""
    out = []
    for c in cons:
        _, final = gen_fn(np.repeat(c[None], n_samples, 0))
        out.append(len({final_vector(f) for f in final}) / n_samples)
    return float(np.mean(out))


DIV_CONS = TEST_CONS[:50]
RESULTS["eval_vectors"]["cgan_snap"]["diversite"] = vector_diversity(METHODS["cgan_snap"], DIV_CONS)
RESULTS["eval_vectors"]["prior"]["diversite"] = vector_diversity(method_prior, DIV_CONS)
RESULTS["eval_vectors"]["aleatoire"]["diversite"] = vector_diversity(method_random, DIV_CONS)

if RUN_ABLATION:
    RESULTS["ablation_lambda"] = {}
    for lam in CFG["abl_lambdas"]:
        set_seed(SEED)
        G_a, D_a = ConditionalGenerator().to(device), ConditionalDiscriminator().to(device)
        h = train_cgan(G_a, D_a, cond_dataloader, device, epochs=CFG["abl_epochs"],
                       lr=CFG["lr"], lambda_constraint=lam, verbose=False)
        r = eval_vectors(method_cgan(G_a, False), TEST_CONS)
        r["plausibilite_snap"] = eval_vectors(method_cgan(G_a, True), TEST_CONS)["plausibilite"]
        r["diversite"] = vector_diversity(method_cgan(G_a, True), DIV_CONS)
        r["d_acc_final"] = h["d_acc"][-1]
        RESULTS["ablation_lambda"][str(lam)] = r
        print(f"lambda={lam:5.1f} | respect brut {r['respect_brut']:.3f} | plausibilite "
              f"{r['plausibilite_snap']:.3f} | diversite {r['diversite']:.3f} | "
              f"D_acc {r['d_acc_final']*100:.1f}%")

    lams = CFG["abl_lambdas"]; A = RESULTS["ablation_lambda"]
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    xs = range(len(lams))
    ax.plot(xs, [A[str(l)]["respect_brut"] for l in lams], "o-", label="Respect brut des contraintes")
    ax.plot(xs, [A[str(l)]["plausibilite_snap"] for l in lams], "s-", label="Plausibilité (après snap)")
    ax.plot(xs, [A[str(l)]["diversite"] for l in lams], "^-", label="Diversité")
    ax.set_xticks(list(xs)); ax.set_xticklabels([str(l) for l in lams])
    ax.set_xlabel("λ (poids de la perte de contraintes)"); ax.set_ylim(0, 1.05)
    ax.grid(alpha=.3); ax.legend(fontsize=8, loc="lower right")
    plt.tight_layout(); plt.savefig(os.path.join(EXPORT_DIR, "fig_ablation_lambda.png"), dpi=200)
    plt.show()

print("\n[Cellule 11 OK]")

## Cellule 12 — Split Learning : implémentation

In [ ]:
# =====================================================================
# CELLULE 12 - SPLIT LEARNING : IMPLEMENTATION (U-shaped, K clients)
# =====================================================================
# Decoupage du cGAN entre K clients (agences, donnees privees) et un
# serveur partage :
#
#   CLIENT k : E_k  encodeur de condition   c (18) -> e (32)   [smashed]
#              B_k  bas du discriminateur  (v, c)  -> s (64)   [smashed]
#              perte de contraintes (necessite c, donc calculee chez k)
#   SERVEUR  : G_s  generateur             (z, e)  -> v_hat (9)
#              T    haut du discriminateur  s      -> proba
#
# Les vecteurs reels v et les conditions c ne quittent JAMAIS le client.
# Ce qui traverse la frontiere : e, s, v_hat (genere, non prive) et les
# gradients correspondants. La frontiere est materialisee par .detach()
# + renvoi explicite des gradients : c'est exactement le protocole qu'on
# aurait sur un reseau, execute ici dans un seul processus. La classe
# Channel compte les octets echanges dans chaque sens.
#
# Variantes d'entrainement :
#   sync="none"   : Split Learning "vanilla" multi-clients, modules
#                   client prives (chaque agence garde E_k, B_k)
#   sync="fedavg" : SplitFed (Thapa et al., 2022) - les modules client
#                   sont moyennes (FedAvg) a la fin de chaque epoque
#   noise_std > 0 : bruit gaussien ajoute aux smashed data avant envoi
#                   (defense simple contre l'inversion, cf. cellule 14)
# =====================================================================

import copy
from collections import defaultdict

SMASH_E, SMASH_S = 32, 64


class ClientCondEncoder(nn.Module):
    def __init__(self, cond_dim=COND_DIM, out=SMASH_E):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(cond_dim, 64), nn.LeakyReLU(0.2),
                                 nn.Linear(64, out), nn.LeakyReLU(0.2))

    def forward(self, c):
        return self.net(c)


class ServerGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, e_dim=SMASH_E, out_dim=VECTOR_DIM):
        super().__init__()
        self.z_dim = latent_dim
        self.model = nn.Sequential(
            nn.Linear(latent_dim + e_dim, 128), nn.LeakyReLU(0.2), nn.BatchNorm1d(128),
            nn.Linear(128, 128), nn.LeakyReLU(0.2), nn.BatchNorm1d(128),
            nn.Linear(128, 64), nn.LeakyReLU(0.2),
            nn.Linear(64, out_dim), nn.Sigmoid())

    def forward(self, z, e):
        return self.model(torch.cat([z, e], dim=1))


class ClientDiscBottom(nn.Module):
    def __init__(self, vector_dim=VECTOR_DIM, cond_dim=COND_DIM, out=SMASH_S):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vector_dim + cond_dim, 128), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(128, out), nn.LeakyReLU(0.2))

    def forward(self, v, c):
        return self.net(torch.cat([v, c], dim=1))


class ServerDiscTop(nn.Module):
    def __init__(self, s_dim=SMASH_S):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(0.3), nn.Linear(s_dim, 64), nn.LeakyReLU(0.2),
                                 nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, s):
        return self.net(s)


class SplitGeneratorView(nn.Module):
    """G(z, c) = G_s(z, E_k(c)) : vue 'inference' du point de vue du client k.
    Compatible avec generate_counts / plan_from_prompt."""
    def __init__(self, E, Gs):
        super().__init__()
        self.E, self.Gs, self.z_dim = E, Gs, Gs.z_dim

    def forward(self, z, c):
        return self.Gs(z, self.E(c))


class Channel:
    """Frontiere client <-> serveur : detache les tenseurs et compte les octets."""
    def __init__(self):
        self.bytes = defaultdict(int)

    def _pass(self, t, kind, grad):
        self.bytes[kind] += t.numel() * t.element_size()
        x = t.detach()
        return x.requires_grad_(True) if grad else x

    def up(self, t, kind, grad=False):      # client -> serveur
        return self._pass(t, "up_" + kind, grad)

    def down(self, t, kind, grad=False):    # serveur -> client
        return self._pass(t, "down_" + kind, grad)

    def total(self):
        return sum(self.bytes.values())


class SplitClient:
    def __init__(self, name, vectors, lr=CFG["lr"]):
        self.name, self.vectors = name, vectors
        self.E, self.B = ClientCondEncoder().to(device), ClientDiscBottom().to(device)
        self.opt_E = torch.optim.Adam(self.E.parameters(), lr=lr, betas=(0.5, 0.999))
        self.opt_B = torch.optim.Adam(self.B.parameters(), lr=lr, betas=(0.5, 0.999))
        self.loader = DataLoader(ConditionalRoomDataset(vectors, MAX_VEC),
                                 batch_size=CFG["batch_size"], shuffle=True,
                                 drop_last=True, num_workers=0)


class SplitServer:
    def __init__(self, lr=CFG["lr"]):
        self.G, self.T = ServerGenerator().to(device), ServerDiscTop().to(device)
        self.opt_G = torch.optim.Adam(self.G.parameters(), lr=lr, betas=(0.5, 0.999))
        self.opt_T = torch.optim.Adam(self.T.parameters(), lr=lr, betas=(0.5, 0.999))


def _noisy(t, std):
    return t + std * torch.randn_like(t) if std > 0 else t


def split_step(cl, sv, target, cond, lam, ch, noise_std, bce):
    """Un pas d'entrainement (D puis G) pour le client cl."""
    b = target.size(0)
    real_lbl = torch.full((b, 1), 0.9, device=device)
    fake_lbl = torch.full((b, 1), 0.1, device=device)

    # ============ Phase D ============
    with torch.no_grad():                                     # [client]
        e_srv = ch.up(_noisy(cl.E(cond), noise_std), "e")
        v_hat = sv.G(torch.randn(b, sv.G.z_dim, device=device), e_srv)   # [serveur]
    v_hat_cli = ch.down(v_hat, "v_hat")

    cl.opt_B.zero_grad(); sv.opt_T.zero_grad()
    s_real = _noisy(cl.B(target, cond), noise_std)            # [client]
    s_fake = _noisy(cl.B(v_hat_cli, cond), noise_std)
    s_real_srv, s_fake_srv = ch.up(s_real, "s", True), ch.up(s_fake, "s", True)
    d_loss = bce(sv.T(s_real_srv), real_lbl) + bce(sv.T(s_fake_srv), fake_lbl)   # [serveur]
    d_loss.backward(); sv.opt_T.step()
    torch.autograd.backward([s_real, s_fake],                 # [client]
                            [ch.down(s_real_srv.grad, "grad_s"),
                             ch.down(s_fake_srv.grad, "grad_s")])
    cl.opt_B.step()

    # ============ Phase G ============
    cl.opt_E.zero_grad(); sv.opt_G.zero_grad()
    e = _noisy(cl.E(cond), noise_std)                         # [client]
    e_srv = ch.up(e, "e", True)
    v_hat = sv.G(torch.randn(b, sv.G.z_dim, device=device), e_srv)   # [serveur]
    v_cli = ch.down(v_hat, "v_hat", True)
    s = _noisy(cl.B(v_cli, cond), noise_std)                  # [client]
    s_srv = ch.up(s, "s", True)
    g_adv = bce(sv.T(s_srv), real_lbl)                        # [serveur]
    g_adv.backward()             # grads de T ignores (remis a zero en phase D)
    g_con = constraint_violation_loss(v_cli, cond)            # [client] (c est prive)
    torch.autograd.backward([s, lam * g_con], [ch.down(s_srv.grad, "grad_s"), None])
    v_hat.backward(ch.up(v_cli.grad, "grad_v_hat"))           # [serveur]
    sv.opt_G.step()
    e.backward(ch.down(e_srv.grad, "grad_e"))                 # [client]
    cl.opt_E.step()
    return d_loss.item(), g_adv.item(), g_con.item()


def fedavg_clients(clients):
    """SplitFed : moyenne ponderee des modules client (E, B)."""
    w = np.array([len(c.vectors) for c in clients], dtype=float); w /= w.sum()
    for attr in ("E", "B"):
        sds = [getattr(c, attr).state_dict() for c in clients]
        avg = {k: sum(wi * sd[k].float() for wi, sd in zip(w, sds)) for k in sds[0]}
        for c in clients:
            getattr(c, attr).load_state_dict(avg)


def train_split(clients, server, epochs, lam=CFG["lambda"], sync="fedavg",
                noise_std=0.0, log_every=10, label=""):
    ch, bce = Channel(), nn.BCELoss()
    hist = {"d_loss": [], "g_adv": [], "g_constraint": [], "bytes": []}
    t0 = time.time()
    for ep in range(epochs):
        for c in clients:
            c.E.train(); c.B.train()
        server.G.train(); server.T.train()
        iters = [iter(c.loader) for c in clients]
        active = list(range(len(clients)))
        ep_d, ep_a, ep_c = [], [], []
        while active:                       # round-robin entre clients
            for k in list(active):
                try:
                    target, cond = next(iters[k])
                except StopIteration:
                    active.remove(k); continue
                d, a, cc = split_step(clients[k], server, target.to(device), cond.to(device),
                                      lam, ch, noise_std, bce)
                ep_d.append(d); ep_a.append(a); ep_c.append(cc)
        if sync == "fedavg" and len(clients) > 1:
            fedavg_clients(clients)
        hist["d_loss"].append(float(np.mean(ep_d))); hist["g_adv"].append(float(np.mean(ep_a)))
        hist["g_constraint"].append(float(np.mean(ep_c))); hist["bytes"].append(ch.total())
        if (ep + 1) % log_every == 0 or ep == 0:
            print(f"  {label} ep {ep+1:>3}/{epochs} | D={hist['d_loss'][-1]:.3f} | "
                  f"G_adv={hist['g_adv'][-1]:.3f} | G_con={hist['g_constraint'][-1]:.4f} | "
                  f"{ch.total()/1e6:.1f} Mo | {time.time()-t0:.0f}s")
    hist["time_s"] = time.time() - t0
    hist["bytes_by_kind"] = dict(ch.bytes)
    return hist


# --- Test unitaire : la frontiere ne laisse pas passer de gradient ----
_cl = SplitClient("test", train_vectors[:256]); _sv = SplitServer()
_t, _c = next(iter(_cl.loader))
_ch = Channel()
split_step(_cl, _sv, _t.to(device), _c.to(device), 1.0, _ch, 0.0, nn.BCELoss())
assert all(p.grad is not None for p in _cl.E.parameters()), "E ne recoit pas de gradient"
assert all(p.grad is not None for p in _sv.G.parameters()), "G_s ne recoit pas de gradient"
print("Octets echanges pour 1 batch :", {k: v for k, v in _ch.bytes.items()})
print(f"Parametres client (E+B): {count_params(_cl.E)+count_params(_cl.B):,} | "
      f"serveur (G_s+T): {count_params(_sv.G)+count_params(_sv.T):,}")
del _cl, _sv, _t, _c, _ch
print("\n[Cellule 12 OK]")

## Cellule 13 — Split Learning : expériences
Centralisé vs local seul vs SL vs SplitFed (± bruit), données non-IID réparties sur 3 agences.

In [ ]:
# =====================================================================
# CELLULE 13 - SPLIT LEARNING : EXPERIENCES
# =====================================================================
# K = 3 agences aux donnees NON-IID (repartition par nombre de chambres) :
#   A "petits logements" (0-1 ch.), B "familial" (2 ch.), C "grands" (3-4 ch.)
#   70 % des programmes vont a l'agence "naturelle", 30 % ailleurs.
#
# Configurations comparees (meme budget d'epoques) :
#   centralise   : toutes les donnees reunies (borne haute, pas de confidentialite)
#   local        : chaque agence entraine seule son cGAN (pas de collaboration)
#   SL           : Split Learning multi-clients, modules client prives
#   SplitFed     : Split Learning + FedAvg des modules client
#   SplitFed+bruit : idem avec bruit gaussien sur les smashed data
#
# Evaluation sur les conditions du jeu de test : globalement, sur le
# segment "propre" de chaque agence et sur les segments "etrangers".
# =====================================================================

def home_client(v):
    nc = int(v[GENERATIVE_ROOMS.index("chambre")])
    return 0 if nc <= 1 else (1 if nc == 2 else 2)


CLIENT_NAMES = ["A (0-1 ch.)", "B (2 ch.)", "C (3-4 ch.)"]


def partition_non_iid(vectors, k=3, p_home=0.7, seed=SEED):
    r = np.random.default_rng(seed)
    parts = [[] for _ in range(k)]
    for v in vectors:
        h = home_client(v)
        dest = h if r.random() < p_home else r.choice([j for j in range(k) if j != h])
        parts[dest].append(v)
    return [torch.stack(p) for p in parts]


def eval_segments(views):
    """views : liste de generateurs (un par client) ou un seul.
    Retourne respect brut / plausibilite : global, segment propre, etranger."""
    if not isinstance(views, list):
        views = [views] * CFG["sl_clients"]
    homes = np.array([home_client(np.where(c >= 0, c, 0)) if c[GENERATIVE_ROOMS.index("chambre")] >= 0
                      else -1 for c in TEST_CONS])
    res = {"global": [], "propre": [], "etranger": []}
    for k, G_k in enumerate(views):
        set_seed(SEED)
        raw, final = generate_counts(G_k, TEST_CONS, MAX_VEC, snap=True)
        resp = np.array([respect_rate(raw[i], TEST_CONS[i]) for i in range(len(TEST_CONS))])
        plaus = np.array([final_vector(final[i]) in SUPPORT for i in range(len(TEST_CONS))])
        res["global"].append((resp.mean(), plaus.mean()))
        own, other = homes == k, (homes >= 0) & (homes != k)
        res["propre"].append((resp[own].mean(), plaus[own].mean()))
        res["etranger"].append((resp[other].mean(), plaus[other].mean()))
    return {seg: {"respect_brut": float(np.mean([a for a, _ in v])),
                  "plausibilite": float(np.mean([b for _, b in v]))} for seg, v in res.items()}


if RUN_SPLIT:
    K, EP = CFG["sl_clients"], CFG["sl_epochs"]
    parts = partition_non_iid(train_vectors, K)
    RESULTS["split"] = {"clients": {CLIENT_NAMES[k]: len(p) for k, p in enumerate(parts)},
                        "epochs": EP, "runs": {}}
    print("Repartition:", RESULTS["split"]["clients"])
    SPLIT_MODELS, SPLIT_HISTS = {}, {}

    def run_split(label, sync, noise=0.0):
        set_seed(SEED)
        clients = [SplitClient(CLIENT_NAMES[k], parts[k]) for k in range(K)]
        server = SplitServer()
        h = train_split(clients, server, EP, sync=sync, noise_std=noise, label=label)
        views = [SplitGeneratorView(c.E, server.G) for c in clients]
        SPLIT_MODELS[label], SPLIT_HISTS[label] = (clients, server, views), h
        return views, h

    # --- 1. Centralise au meme budget d'epoques (checkpoint de la cellule 8)
    ck = os.path.join(MODEL_DIR, f"cgan_ep{EP:03d}.pt")
    G_ref = ConditionalGenerator().to(device)
    if os.path.exists(ck):
        G_ref.load_state_dict(torch.load(ck, map_location=device, weights_only=False)["G"])
    else:
        G_ref = G_cond
    RESULTS["split"]["runs"]["centralise"] = {"eval": eval_segments(G_ref)}

    # --- 2. Local seul (chaque agence, sans collaboration)
    local_views, local_time = [], 0.0
    for k in range(K):
        set_seed(SEED)
        cl, sv = SplitClient(CLIENT_NAMES[k], parts[k]), SplitServer()
        h = train_split([cl], sv, EP, sync="none", label=f"local {k}")
        local_views.append(SplitGeneratorView(cl.E, sv.G)); local_time += h["time_s"]
    RESULTS["split"]["runs"]["local"] = {"eval": eval_segments(local_views), "time_s": local_time}

    # --- 3. SL (modules client prives), 4. SplitFed, 5. SplitFed + bruit
    configs = [("SL", "none", 0.0), ("SplitFed", "fedavg", 0.0)]
    if RUN_NOISE:
        configs.append(("SplitFed+bruit", "fedavg", CFG["sl_noise_std"]))
    for label, sync, noise in configs:
        views, h = run_split(label, sync, noise)
        n_batches = sum(len(DataLoader(p, batch_size=CFG["batch_size"], drop_last=True)) for p in parts)
        RESULTS["split"]["runs"][label] = {
            "eval": eval_segments(views if sync == "none" else views[0]),
            "time_s": h["time_s"], "bytes_total": h["bytes"][-1],
            "bytes_per_epoch": h["bytes"][-1] / EP,
            "bytes_per_sample": h["bytes"][-1] / EP / (n_batches * CFG["batch_size"]),
            "bytes_by_kind": h["bytes_by_kind"], "noise_std": noise,
            "final_g_constraint": h["g_constraint"][-1]}

    # --- Couts compares (par epoque, toutes agences confondues) ---------
    _c, _s = SplitClient("x", parts[0][:200]), SplitServer()
    p_client = count_params(_c.E) + count_params(_c.B)
    p_server = count_params(_s.G) + count_params(_s.T)
    RESULTS["split"]["params_client"] = p_client
    RESULTS["split"]["params_server"] = p_server
    RESULTS["split"]["client_param_share"] = p_client / (p_client + p_server)
    # FedAvg classique : chaque client envoie et recoit le modele complet a chaque tour
    RESULTS["split"]["fl_bytes_per_epoch"] = 2 * K * (p_client + p_server) * 4
    RESULTS["split"]["splitfed_sync_bytes_per_epoch"] = 2 * K * p_client * 4
    RESULTS["split"]["raw_data_bytes"] = int(train_vectors.numel() * 4)
    del _c, _s

    # --- Tableau recapitulatif -----------------------------------------
    print(f"\n{'config':16s} | {'global':>15s} | {'segment propre':>15s} | {'segment etranger':>16s}")
    print(f"{'':16s} | {'resp.  plaus.':>15s} | {'resp.  plaus.':>15s} | {'resp.  plaus.':>16s}")
    for name, r in RESULTS["split"]["runs"].items():
        e = r["eval"]
        print(f"{name:16s} | " + " | ".join(f"{e[s]['respect_brut']:.3f}  {e[s]['plausibilite']:.3f}"
                                           for s in ("global", "propre", "etranger")))
    sf = RESULTS["split"]["runs"]["SplitFed"]
    print(f"\nSplitFed : {sf['bytes_per_epoch']/1e6:.2f} Mo/epoque de smashed data + gradients "
          f"(+ {RESULTS['split']['splitfed_sync_bytes_per_epoch']/1e6:.2f} Mo de synchro) | "
          f"FedAvg complet : {RESULTS['split']['fl_bytes_per_epoch']/1e6:.2f} Mo/epoque | "
          f"part des parametres cote client : {RESULTS['split']['client_param_share']:.1%}")

    # --- Figure : convergence des variantes ----------------------------
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    hc = json.load(open(os.path.join(EXPORT_DIR, "hist_central.json")))
    ax[0].plot(hc["g_constraint"][:EP], "k--", label="centralisé")
    for label, h in SPLIT_HISTS.items():
        ax[0].plot(h["g_constraint"], label=label)
        ax[1].plot(h["d_loss"], label=label)
    ax[1].plot(hc["d_loss"][:EP], "k--", label="centralisé")
    ax[0].set_yscale("log"); ax[0].set_title("Violation des contraintes (MSE)")
    ax[1].set_title("Perte du discriminateur")
    for a in ax:
        a.set_xlabel("époque"); a.grid(alpha=.3); a.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(EXPORT_DIR, "fig_split_training.png"), dpi=200)
    plt.show()

    # --- Les plans produits par le modele SplitFed ----------------------
    G_sf = SPLIT_MODELS["SplitFed"][2][0]
    rows = eval_plans(method_cgan(G_sf, True), BENCH_PROMPTS, max(2, N_VAR // 5))
    RESULTS["split"]["splitfed_plans"] = {k: float(np.mean([r[k] for r in rows])) for k in
                                          ["respect_plan", "plausibilite", "connectivite",
                                           "portes_legales", "adjacence", "t_total"]}
    print("Plans SplitFed:", {k: round(v, 3) for k, v in RESULTS["split"]["splitfed_plans"].items()})
    _ = plan("T3 avec balcon", G=G_sf, seed=SEED)

print("\n[Cellule 13 OK]")

## Cellule 14 — Fuite d'information des *smashed data*

In [ ]:
# =====================================================================
# CELLULE 14 - FUITE D'INFORMATION DES SMASHED DATA (attaque d'inversion)
# =====================================================================
# Le Split Learning ne transmet pas les donnees brutes, mais les smashed
# data peuvent en reveler une partie (cf. attaques PCAT, Gao et al. 2023).
# Scenario : un serveur curieux dispose de quelques paires
# (donnee, smashed data) ayant fuite (20 %) et entraine un inverseur
#   s -> v   (retrouver le programme reel depuis B_k(v, c))
# On mesure le taux de programmes EXACTEMENT reconstruits sur le reste,
# compare a une baseline sans information (programme le plus frequent).
# =====================================================================


def inversion_attack(B, vectors, noise_std=0.0, leak=0.2, epochs=2000, seed=SEED):
    set_seed(seed)
    B.eval()
    v_norm = (vectors / MAX_VEC).to(device)
    mask = (torch.rand_like(v_norm) > 0.5).float()
    cond = torch.cat([v_norm * mask, mask], 1)
    with torch.no_grad():
        s = _noisy(B(v_norm, cond), noise_std)
    n_leak = int(leak * len(s))
    inv = nn.Sequential(nn.Linear(s.shape[1], 128), nn.ReLU(), nn.Linear(128, 128),
                        nn.ReLU(), nn.Linear(128, VECTOR_DIM), nn.Sigmoid()).to(device)
    opt = torch.optim.Adam(inv.parameters(), lr=1e-3)
    for _ in range(epochs):
        opt.zero_grad()
        loss = ((inv(s[:n_leak]) - v_norm[:n_leak]) ** 2).mean()
        loss.backward(); opt.step()
    with torch.no_grad():
        rec = torch.round(inv(s[n_leak:]) * MAX_VEC.to(device)).cpu()
    true = vectors[n_leak:]
    exact = (rec == true).all(1).float().mean().item()
    per_entry = (rec == true).float().mean().item()
    mode = torch.tensor(Counter(tuple(v.tolist()) for v in vectors[:n_leak]).most_common(1)[0][0])
    base_exact = (true == mode).all(1).float().mean().item()
    base_entry = (true == mode).float().mean().item()
    return {"exact": exact, "par_entree": per_entry,
            "baseline_exact": base_exact, "baseline_par_entree": base_entry}


if RUN_SPLIT:
    RESULTS["leakage"] = {}
    vic = test_vectors[:min(2000, len(test_vectors))]
    for label in SPLIT_MODELS:
        clients, _, _ = SPLIT_MODELS[label]
        noise = RESULTS["split"]["runs"][label]["noise_std"]
        r = inversion_attack(clients[0].B, vic, noise_std=noise,
                             epochs=50 if QUICK else 2000)
        RESULTS["leakage"][label] = r
        print(f"{label:16s} | reconstruction exacte {r['exact']:.3f} (baseline {r['baseline_exact']:.3f})"
              f" | par entree {r['par_entree']:.3f} (baseline {r['baseline_par_entree']:.3f})")

print("\n[Cellule 14 OK]")

## Cellule 15 — Export (JSON, figures, macros LaTeX pour l'article)

In [ ]:
# =====================================================================
# CELLULE 15 - EXPORT DES RESULTATS
# =====================================================================
# Produit dans EXPORT_DIR :
#   resultats.json          : tous les chiffres
#   resultats_macros.tex    : macros LaTeX (\input dans l'article) ->
#                             l'article se met a jour en relancant le notebook
#   fig_*.png               : figures de l'article
#   archigan_export.zip     : le tout, a telecharger
# =====================================================================

import shutil

json.dump(RESULTS, open(os.path.join(EXPORT_DIR, "resultats.json"), "w"),
          indent=2, ensure_ascii=False, default=float)


def fr(x, d=3):
    """Nombre au format francais pour LaTeX : 0.923 -> 0{,}923"""
    return f"{x:.{d}f}".replace(".", "{,}")


def pct(x, d=1):
    return f"{100*x:.{d}f}".replace(".", "{,}") + "\\,\\%"


def macro_name(*parts):
    """Les noms de macros LaTeX n'acceptent que des lettres."""
    s = "".join(p.title() for p in parts)
    for a, b in {"0": "Zero", "1": "Un", "2": "Deux", "3": "Trois", "4": "Quatre",
                 "5": "Cinq", "6": "Six", "7": "Sept", "8": "Huit", "9": "Neuf"}.items():
        s = s.replace(a, b)
    return "res" + "".join(ch for ch in s if ch.isalpha())


M = {}
D = RESULTS["dataset"]
M[macro_name("ds", "n")] = f"{D['n']:,}".replace(",", "\\,")
M[macro_name("ds", "unique")] = str(D["n_unique"])
M[macro_name("ds", "mean", "rooms")] = fr(D["mean_rooms"], 2)
M[macro_name("ds", "unique", "pieces")] = str(D["n_unique_pieces"])
if RESULTS.get("dataset_gen", {}).get("acceptance"):
    M[macro_name("ds", "acceptance")] = pct(RESULTS["dataset_gen"]["acceptance"])
C = RESULTS["central_training"]
M[macro_name("params", "g")] = f"{C['params_G']:,}".replace(",", "\\,")
M[macro_name("params", "d")] = f"{C['params_D']:,}".replace(",", "\\,")
M[macro_name("train", "minutes")] = fr(C["time_s"] / 60, 1)
M[macro_name("train", "dacc")] = pct(C["final"]["d_acc"])
for meth, r in RESULTS["eval_vectors"].items():
    for k, v in r.items():
        M[macro_name("vec", meth, k)] = pct(v)
for meth, agg in RESULTS["eval_plans"].items():
    for k, v in agg.items():
        if v["mean"] != v["mean"]:          # nan (metrique non applicable)
            continue
        if k.startswith("t_"):
            M[macro_name("plan", meth, k)] = fr(1000 * v["mean"], 1)      # en ms
        elif k == "portes_piece_privee":
            M[macro_name("plan", meth, k)] = fr(v["mean"], 2)
        else:
            M[macro_name("plan", meth, k)] = pct(v["mean"])
            M[macro_name("plan", meth, k, "ci")] = fr(100 * v["ci95"], 1)
tt = RESULTS["eval_plans"]["cgan_snap"]["t_total"]
M[macro_name("plan", "time", "min")] = fr(1000 * tt["min"], 1)
M[macro_name("plan", "time", "max")] = fr(1000 * tt["max"], 1)
M[macro_name("plan", "time", "median")] = fr(1000 * tt["median"], 1)
for lam, r in RESULTS.get("ablation_lambda", {}).items():
    for k in ("respect_brut", "plausibilite_snap", "diversite"):
        M[macro_name("abl", "lambda", lam.replace(".", "p"), k)] = pct(r[k])
if "split" in RESULTS:
    S = RESULTS["split"]
    for run, r in S["runs"].items():
        for seg, e in r["eval"].items():
            for k, v in e.items():
                M[macro_name("sl", run, seg, k)] = pct(v)
        if "bytes_per_epoch" in r:
            M[macro_name("sl", run, "mo", "epoch")] = fr(r["bytes_per_epoch"] / 1e6, 2)
            M[macro_name("sl", run, "octets", "sample")] = fr(r["bytes_per_sample"], 0)
            M[macro_name("sl", run, "minutes")] = fr(r["time_s"] / 60, 1)
    M[macro_name("sl", "fl", "mo", "epoch")] = fr(S["fl_bytes_per_epoch"] / 1e6, 2)
    M[macro_name("sl", "sync", "mo", "epoch")] = fr(S["splitfed_sync_bytes_per_epoch"] / 1e6, 2)
    M[macro_name("sl", "client", "share")] = pct(S["client_param_share"])
    M[macro_name("sl", "params", "client")] = f"{S['params_client']:,}".replace(",", "\\,")
    M[macro_name("sl", "params", "server")] = f"{S['params_server']:,}".replace(",", "\\,")
    M[macro_name("sl", "epochs")] = str(S["epochs"])
    for run, r in RESULTS.get("leakage", {}).items():
        for k, v in r.items():
            M[macro_name("leak", run, k)] = pct(v)

with open(os.path.join(EXPORT_DIR, "resultats_macros.tex"), "w") as f:
    f.write("% Genere automatiquement par le notebook ArchiGAN-SL (cellule 15).\n")
    f.write(f"% device={RESULTS['gpu']} quick={QUICK}\n")
    for k, v in sorted(M.items()):
        f.write(f"\\newcommand{{\\{k}}}{{{v}}}\n")
print(f"{len(M)} macros LaTeX ecrites.")

zip_path = shutil.make_archive(os.path.join(BASE_DIR, "archigan_export"), "zip", EXPORT_DIR)
print(f"Archive : {zip_path}")
if IN_COLAB:
    colab_files.download(zip_path)
print("\n[Cellule 15 OK]")

## Bac à sable
Essayez vos propres prompts :

In [ ]:
plan("T3 avec balcon et un bureau", n=2)